In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/diagnostics.py ──
"""DL Diagnostics Toolkit — clinical instruments for deep-network training.

Wraps PyTorch forward/backward hooks into a student-friendly API that
surfaces the four failure modes professionals must recognise:

    1. Stethoscope  — loss-curve shape (under/over-fit, instability)
    2. Blood Test   — gradient flow per layer (vanishing / exploding)
    3. X-Ray        — activation statistics & dead neurons
    4. Prescription — automated diagnosis with actionable suggestions

Typical use inside an exercise::


    with DLDiagnostics(model) as diag:
        diag.track_gradients()
        diag.track_activations()
        diag.track_dead_neurons()
        for epoch in range(epochs):
            for batch in dataloader:
                loss = train_step(model, batch)
                diag.record_batch(loss=loss.item(),
                                  lr=optimizer.param_groups[0]["lr"])
            diag.record_epoch(val_loss=evaluate(model, val_loader))
        diag.plot_training_dashboard().show()
        diag.report()

All DataFrames are Polars. All plots are Plotly. No matplotlib, no pandas.
"""

import logging
import math
from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import plotly.graph_objects as go
import polars as pl
import torch
import torch.nn as nn
from plotly.subplots import make_subplots
from torch.utils.data import DataLoader


logger = logging.getLogger(__name__)

# Module names whose outputs are "dead-neuron sensitive" — we track fraction
# of zero outputs for these specifically.
_DEAD_NEURON_SENSITIVE: tuple[type[nn.Module], ...] = (
    nn.ReLU,
    nn.LeakyReLU,
    nn.GELU,
    nn.ELU,
    nn.SiLU,
)

# Module names whose outputs we monitor for activation statistics.
_ACTIVATION_MONITORED: tuple[type[nn.Module], ...] = (
    nn.Linear,
    nn.Conv1d,
    nn.Conv2d,
    nn.Conv3d,
    nn.ReLU,
    nn.LeakyReLU,
    nn.GELU,
    nn.ELU,
    nn.SiLU,
    nn.Tanh,
    nn.Sigmoid,
    nn.BatchNorm1d,
    nn.BatchNorm2d,
    nn.LayerNorm,
)


@dataclass
class _HookHandles:
    """Container for registered hook handles so we can detach cleanly."""

    gradient: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    activation: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    dead_neuron: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    grad_cam: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)

    def all(self) -> list[torch.utils.hooks.RemovableHandle]:
        return self.gradient + self.activation + self.dead_neuron + self.grad_cam


class DLDiagnostics:
    """Clinical instrumentation for PyTorch training loops.

    Collects per-batch time series of gradient norms, activation statistics,
    dead-neuron fractions, and losses; exposes Plotly visualisations and an
    automated report that surfaces overfitting, vanishing gradients, and
    pathological dead-ReLU layers.

    Args:
        model: The ``nn.Module`` to instrument. The model is NOT modified;
            only forward/backward hooks are attached.
        dead_neuron_threshold: Fraction of zero outputs above which a layer
            is flagged as "dead" in :meth:`report`. Defaults to ``0.5``.
        window: Number of recent batches used for dead-neuron statistics.
            Defaults to ``64``.

    Example:
        >>> model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1))
        >>> with DLDiagnostics(model) as diag:
        ...     diag.track_gradients()
        ...     diag.track_activations()
        ...     # ... training loop ...
    """

    def __init__(
        self,
        model: nn.Module,
        *,
        dead_neuron_threshold: float = 0.5,
        window: int = 64,
    ) -> None:
        if not isinstance(model, nn.Module):
            raise TypeError(
                f"DLDiagnostics requires an nn.Module; got {type(model).__name__}"
            )
        if not 0.0 < dead_neuron_threshold < 1.0:
            raise ValueError("dead_neuron_threshold must be in (0, 1)")
        if window < 1:
            raise ValueError("window must be >= 1")

        self.model = model
        self.device = get_device()
        self.dead_neuron_threshold = dead_neuron_threshold
        self.window = window

        # Time series storage — lists of dicts, converted to Polars on demand.
        self._grad_log: list[dict[str, Any]] = []
        self._act_log: list[dict[str, Any]] = []
        self._dead_log: list[dict[str, Any]] = []
        self._batch_log: list[dict[str, Any]] = []
        self._epoch_log: list[dict[str, Any]] = []

        # Running per-layer firing masks for dead-neuron detection.
        # Key: layer name -> tensor of firing counts per neuron (1D).
        self._firing_counts: dict[str, torch.Tensor] = {}
        self._firing_samples: dict[str, int] = {}

        # Counters bound to hook closures so they share scope.
        self._batch_idx = 0
        self._epoch_idx = 0

        self._handles = _HookHandles()
        self._tracking = {"gradients": False, "activations": False, "dead": False}

        # Grad-CAM capture buffers (populated on demand).
        self._gradcam_activation: torch.Tensor | None = None
        self._gradcam_gradient: torch.Tensor | None = None

        logger.info(
            "dldiagnostics.init",
            extra={
                "model_class": type(model).__name__,
                "device": str(self.device),
                "window": window,
            },
        )

    # ── Context-manager support ────────────────────────────────────────────

    def __enter__(self) -> DLDiagnostics:
        return self

    def __exit__(self, exc_type, exc, tb) -> None:  # noqa: D401
        self.detach()

    def __del__(self) -> None:  # pragma: no cover - best-effort cleanup
        try:
            self.detach()
        except Exception:
            # Finalizers MUST NOT raise. Silent cleanup is the documented
            # contract for __del__ in rules/patterns.md.
            pass

    # ── Hook registration ──────────────────────────────────────────────────

    def track_gradients(self) -> DLDiagnostics:
        """Register backward hooks on every trainable parameter.

        Records the L2 norm of each parameter's gradient at every backward
        pass, keyed by parameter name.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["gradients"]:
            return self
        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            handle = param.register_hook(self._make_grad_hook(name))
            self._handles.gradient.append(handle)
        self._tracking["gradients"] = True
        logger.info(
            "dldiagnostics.track_gradients",
            extra={"hooks_registered": len(self._handles.gradient)},
        )
        return self

    def track_activations(self) -> DLDiagnostics:
        """Register forward hooks on monitored submodules.

        Records mean/std/min/max/dead-fraction of activations per layer
        at every forward pass.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["activations"]:
            return self
        for name, module in self.model.named_modules():
            if name == "" or not isinstance(module, _ACTIVATION_MONITORED):
                continue
            handle = module.register_forward_hook(self._make_act_hook(name))
            self._handles.activation.append(handle)
        self._tracking["activations"] = True
        logger.info(
            "dldiagnostics.track_activations",
            extra={"hooks_registered": len(self._handles.activation)},
        )
        return self

    def track_dead_neurons(self) -> DLDiagnostics:
        """Register forward hooks on ReLU-family layers to track dead neurons.

        A "neuron" here is a channel (Conv) or output unit (Linear). The
        rolling fraction of batches where that neuron output zero is tracked.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["dead"]:
            return self
        for name, module in self.model.named_modules():
            if name == "" or not isinstance(module, _DEAD_NEURON_SENSITIVE):
                continue
            handle = module.register_forward_hook(self._make_dead_hook(name))
            self._handles.dead_neuron.append(handle)
        self._tracking["dead"] = True
        logger.info(
            "dldiagnostics.track_dead_neurons",
            extra={"hooks_registered": len(self._handles.dead_neuron)},
        )
        return self

    def detach(self) -> None:
        """Remove ALL registered hooks and release references.

        Safe to call multiple times. Invoked automatically on context exit.
        """
        for handle in self._handles.all():
            try:
                handle.remove()
            except Exception:
                # Hook removal failures are benign (module may already be
                # gone). See rules/zero-tolerance.md Rule 3 carve-out for
                # cleanup paths.
                pass
        self._handles = _HookHandles()
        self._tracking = {k: False for k in self._tracking}
        self._gradcam_activation = None
        self._gradcam_gradient = None

    # ── Recording ─────────────────────────────────────────────────────────

    def record_batch(self, *, loss: float, lr: float | None = None) -> None:
        """Record per-batch scalar training signals.

        Args:
            loss: Training loss for the batch (post-backward).
            lr: Current learning rate (optional; read from optimizer).
        """
        if not math.isfinite(loss):
            logger.warning(
                "dldiagnostics.record_batch.nonfinite_loss",
                extra={"loss": str(loss), "batch": self._batch_idx},
            )
        self._batch_log.append(
            {
                "batch": self._batch_idx,
                "epoch": self._epoch_idx,
                "loss": float(loss),
                "lr": float(lr) if lr is not None else float("nan"),
            }
        )
        self._batch_idx += 1

    def record_epoch(
        self,
        *,
        val_loss: float | None = None,
        train_loss: float | None = None,
        **extra: float,
    ) -> None:
        """Record per-epoch summary metrics.

        Args:
            val_loss: Validation loss at epoch end.
            train_loss: Mean training loss for the epoch. If ``None``, it is
                computed from the batches in this epoch.
            **extra: Any additional scalar metrics to persist.
        """
        if train_loss is None:
            epoch_batches = [
                b for b in self._batch_log if b["epoch"] == self._epoch_idx
            ]
            if epoch_batches:
                train_loss = float(np.mean([b["loss"] for b in epoch_batches]))
        entry = {
            "epoch": self._epoch_idx,
            "train_loss": train_loss if train_loss is not None else float("nan"),
            "val_loss": val_loss if val_loss is not None else float("nan"),
            **{k: float(v) for k, v in extra.items()},
        }
        self._epoch_log.append(entry)
        self._epoch_idx += 1

    # ── Public DataFrame accessors ────────────────────────────────────────

    def gradients_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-layer gradient norms by batch."""
        if not self._grad_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "layer": pl.Utf8,
                    "grad_norm": pl.Float64,
                    "grad_rms": pl.Float64,
                    "update_ratio": pl.Float64,
                }
            )
        return pl.DataFrame(self._grad_log)

    def activations_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-layer activation statistics by batch."""
        if not self._act_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "layer": pl.Utf8,
                    "act_kind": pl.Utf8,
                    "mean": pl.Float64,
                    "std": pl.Float64,
                    "min": pl.Float64,
                    "max": pl.Float64,
                    "dead_fraction": pl.Float64,
                    "inactivity_fraction": pl.Float64,
                }
            )
        return pl.DataFrame(self._act_log)

    def dead_neurons_df(self) -> pl.DataFrame:
        """Polars DataFrame of current per-layer dead-neuron fractions."""
        rows: list[dict[str, Any]] = []
        for name, counts in self._firing_counts.items():
            n_samples = max(self._firing_samples.get(name, 1), 1)
            dead_mask = (counts / n_samples) < 1e-6
            rows.append(
                {
                    "layer": name,
                    "n_neurons": int(counts.numel()),
                    "n_dead": int(dead_mask.sum().item()),
                    "dead_fraction": float(dead_mask.float().mean().item()),
                }
            )
        if not rows:
            return pl.DataFrame(
                schema={
                    "layer": pl.Utf8,
                    "n_neurons": pl.Int64,
                    "n_dead": pl.Int64,
                    "dead_fraction": pl.Float64,
                }
            )
        return pl.DataFrame(rows)

    def batches_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-batch scalars (loss, lr)."""
        if not self._batch_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "epoch": pl.Int64,
                    "loss": pl.Float64,
                    "lr": pl.Float64,
                }
            )
        return pl.DataFrame(self._batch_log)

    def epochs_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-epoch summary metrics."""
        if not self._epoch_log:
            return pl.DataFrame(
                schema={
                    "epoch": pl.Int64,
                    "train_loss": pl.Float64,
                    "val_loss": pl.Float64,
                }
            )
        return pl.DataFrame(self._epoch_log)

    # ── Plots ─────────────────────────────────────────────────────────────

    def plot_loss_curves(self) -> go.Figure:
        """Loss stethoscope: train vs val curves with overfitting callout.

        Returns:
            A Plotly Figure with two traces (train / val) and annotations
            for detected overfitting epoch (if any).
        """
        epochs = self.epochs_df()
        batches = self.batches_df()
        fig = go.Figure()
        if batches.height:
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["loss"].to_list(),
                    mode="lines",
                    name="train (per batch)",
                    line=dict(color="lightblue", width=1),
                    opacity=0.5,
                )
            )
        if epochs.height:
            fig.add_trace(
                go.Scatter(
                    x=epochs["epoch"].to_list(),
                    y=epochs["train_loss"].to_list(),
                    mode="lines+markers",
                    name="train (epoch mean)",
                    line=dict(color="steelblue", width=2),
                )
            )
            if epochs["val_loss"].is_not_null().any():
                fig.add_trace(
                    go.Scatter(
                        x=epochs["epoch"].to_list(),
                        y=epochs["val_loss"].to_list(),
                        mode="lines+markers",
                        name="val",
                        line=dict(color="firebrick", width=2),
                    )
                )
        overfit_epoch = self._detect_overfit_epoch()
        if overfit_epoch is not None:
            fig.add_vline(
                x=overfit_epoch,
                line=dict(color="orange", dash="dash"),
                annotation_text=f"overfitting suspected @ epoch {overfit_epoch}",
                annotation_position="top",
            )
        fig.update_layout(
            title="Loss Curves (Stethoscope)",
            xaxis_title="step",
            yaxis_title="loss",
            template="plotly_white",
            hovermode="x unified",
        )
        return fig

    def plot_gradient_flow(self) -> go.Figure:
        """Blood test: per-layer gradient norm over time.

        One trace per tracked parameter. Layers whose gradient norm collapses
        toward zero are the vanishing-gradient culprits.
        """
        df = self.gradients_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Gradient Flow (Blood Test) — no data",
                template="plotly_white",
            )
            return fig
        for layer in df["layer"].unique().to_list():
            sub = df.filter(pl.col("layer") == layer)
            fig.add_trace(
                go.Scatter(
                    x=sub["batch"].to_list(),
                    y=sub["grad_norm"].to_list(),
                    mode="lines",
                    name=layer,
                    hovertemplate=f"{layer}<br>batch=%{{x}}<br>‖∇‖=%{{y:.3e}}",
                )
            )
        fig.update_layout(
            title="Gradient Flow by Layer (Blood Test)",
            xaxis_title="batch",
            yaxis_title="gradient L2 norm",
            yaxis_type="log",
            template="plotly_white",
        )
        return fig

    def plot_activation_stats(self) -> go.Figure:
        """X-Ray: activation mean ± std per layer over time."""
        df = self.activations_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Activation Statistics (X-Ray) — no data",
                template="plotly_white",
            )
            return fig
        for layer in df["layer"].unique().to_list():
            sub = df.filter(pl.col("layer") == layer)
            fig.add_trace(
                go.Scatter(
                    x=sub["batch"].to_list(),
                    y=sub["mean"].to_list(),
                    mode="lines",
                    name=f"{layer} — mean",
                    hovertemplate=(
                        f"{layer}<br>batch=%{{x}}<br>mean=%{{y:.3f}}<br>"
                        "std=%{customdata:.3f}"
                    ),
                    customdata=sub["std"].to_list(),
                )
            )
        fig.update_layout(
            title="Activation Mean per Layer (X-Ray)",
            xaxis_title="batch",
            yaxis_title="activation mean",
            template="plotly_white",
        )
        return fig

    def plot_dead_neurons(self) -> go.Figure:
        """X-Ray: fraction of dead neurons per ReLU-family layer."""
        df = self.dead_neurons_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Dead Neurons (X-Ray) — no ReLU-family layers tracked",
                template="plotly_white",
            )
            return fig
        colors = [
            "firebrick" if frac > self.dead_neuron_threshold else "steelblue"
            for frac in df["dead_fraction"].to_list()
        ]
        fig.add_trace(
            go.Bar(
                x=df["layer"].to_list(),
                y=df["dead_fraction"].to_list(),
                marker_color=colors,
                text=[
                    f"{int(n)}/{int(t)}" for n, t in zip(df["n_dead"], df["n_neurons"])
                ],
                textposition="outside",
            )
        )
        fig.add_hline(
            y=self.dead_neuron_threshold,
            line=dict(color="orange", dash="dash"),
            annotation_text=f"alert threshold ({self.dead_neuron_threshold:.0%})",
        )
        fig.update_layout(
            title="Dead Neuron Fraction by Layer (X-Ray)",
            xaxis_title="layer",
            yaxis_title="fraction dead",
            yaxis=dict(range=[0, 1]),
            template="plotly_white",
            showlegend=False,
        )
        return fig

    def plot_training_dashboard(self) -> go.Figure:
        """Vital signs: 2x2 dashboard (loss, grad flow, activations, LR)."""
        fig = make_subplots(
            rows=2,
            cols=2,
            subplot_titles=(
                "Loss (Stethoscope)",
                "Gradient Flow (Blood Test)",
                "Activation Mean (X-Ray)",
                "Learning Rate",
            ),
        )

        # (1,1) Loss
        epochs = self.epochs_df()
        batches = self.batches_df()
        if batches.height:
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["loss"].to_list(),
                    mode="lines",
                    name="train loss",
                    line=dict(color="steelblue", width=1),
                ),
                row=1,
                col=1,
            )
        if epochs.height and epochs["val_loss"].is_not_null().any():
            # Place val loss at the last batch of each epoch for alignment.
            val_x = []
            for ep in epochs["epoch"].to_list():
                ep_batches = batches.filter(pl.col("epoch") == ep)
                val_x.append(
                    int(ep_batches["batch"].max()) if ep_batches.height else ep
                )
            fig.add_trace(
                go.Scatter(
                    x=val_x,
                    y=epochs["val_loss"].to_list(),
                    mode="lines+markers",
                    name="val loss",
                    line=dict(color="firebrick"),
                ),
                row=1,
                col=1,
            )

        # (1,2) Gradient flow
        gdf = self.gradients_df()
        if gdf.height:
            for layer in gdf["layer"].unique().to_list():
                sub = gdf.filter(pl.col("layer") == layer)
                fig.add_trace(
                    go.Scatter(
                        x=sub["batch"].to_list(),
                        y=sub["grad_norm"].to_list(),
                        mode="lines",
                        name=layer,
                        showlegend=False,
                    ),
                    row=1,
                    col=2,
                )
            fig.update_yaxes(type="log", row=1, col=2)

        # (2,1) Activation mean
        adf = self.activations_df()
        if adf.height:
            for layer in adf["layer"].unique().to_list():
                sub = adf.filter(pl.col("layer") == layer)
                fig.add_trace(
                    go.Scatter(
                        x=sub["batch"].to_list(),
                        y=sub["mean"].to_list(),
                        mode="lines",
                        name=layer,
                        showlegend=False,
                    ),
                    row=2,
                    col=1,
                )

        # (2,2) LR
        if batches.height and batches["lr"].is_not_null().any():
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["lr"].to_list(),
                    mode="lines",
                    name="lr",
                    line=dict(color="darkgreen"),
                    showlegend=False,
                ),
                row=2,
                col=2,
            )

        fig.update_layout(
            title="Training Dashboard (Vital Signs)",
            template="plotly_white",
            height=720,
        )
        return fig

    def plot_lr_vs_loss(self) -> go.Figure:
        """Plot LR vs loss (useful after an :meth:`lr_range_test`)."""
        df = self.batches_df()
        fig = go.Figure()
        if df.height == 0 or df["lr"].is_null().all():
            fig.update_layout(title="LR vs Loss — no data", template="plotly_white")
            return fig
        fig.add_trace(
            go.Scatter(
                x=df["lr"].to_list(),
                y=df["loss"].to_list(),
                mode="lines",
                line=dict(color="steelblue"),
            )
        )
        fig.update_layout(
            title="Learning Rate vs Loss",
            xaxis_title="learning rate",
            yaxis_title="loss",
            xaxis_type="log",
            template="plotly_white",
        )
        return fig

    def plot_weight_distributions(self) -> go.Figure:
        """Histogram of weight values, one trace per parameter tensor."""
        fig = go.Figure()
        for name, param in self.model.named_parameters():
            if not param.requires_grad or param.ndim == 0:
                continue
            values = param.detach().cpu().flatten().numpy()
            fig.add_trace(go.Histogram(x=values, name=name, opacity=0.6))
        fig.update_layout(
            title="Weight Distributions",
            xaxis_title="value",
            yaxis_title="count",
            barmode="overlay",
            template="plotly_white",
        )
        return fig

    def plot_gradient_norms(self) -> go.Figure:
        """Mean gradient norm per layer across the run (bar chart)."""
        df = self.gradients_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(title="Gradient Norms — no data", template="plotly_white")
            return fig
        summary = df.group_by("layer").agg(
            pl.col("grad_norm").mean().alias("mean_grad")
        )
        summary = summary.sort("mean_grad")
        fig.add_trace(
            go.Bar(
                x=summary["layer"].to_list(),
                y=summary["mean_grad"].to_list(),
                marker_color="steelblue",
            )
        )
        fig.update_layout(
            title="Mean Gradient Norm per Layer",
            xaxis_title="layer",
            yaxis_title="mean ‖∇‖",
            yaxis_type="log",
            template="plotly_white",
        )
        return fig

    # ── Automated report ──────────────────────────────────────────────────

    def report(self) -> dict[str, Any]:
        """Print a human-readable diagnosis and return findings as a dict.

        Findings covered:
            * Gradient flow (vanishing / exploding / healthy)
            * Dead neurons (per-layer ReLU-family)
            * Loss trend (overfitting, underfitting, instability)

        Returns:
            A dict with keys ``gradient_flow``, ``dead_neurons``, ``loss_trend``
            each mapping to a dict with ``severity`` and ``message``.
        """
        findings: dict[str, Any] = {}

        # 1. Gradient flow — uses SCALE-INVARIANT per-element RMS (grad_rms)
        # and update_ratio (‖∇W‖/‖W‖), matching slide 5F thresholds.
        gdf = self.gradients_df()
        if gdf.height and "grad_rms" in gdf.columns:
            stats = gdf.group_by("layer").agg(
                pl.col("grad_rms").mean().alias("rms"),
                pl.col("update_ratio").mean().alias("ur"),
            )
            min_rms_raw = stats["rms"].min()
            max_rms_raw = stats["rms"].max()
            min_ur_raw = stats["ur"].min()
            max_ur_raw = stats["ur"].max()
            min_rms = (
                float(min_rms_raw) if isinstance(min_rms_raw, (int, float)) else None
            )
            max_rms = (
                float(max_rms_raw) if isinstance(max_rms_raw, (int, float)) else None
            )
            min_ur = float(min_ur_raw) if isinstance(min_ur_raw, (int, float)) else 0.0
            max_ur = float(max_ur_raw) if isinstance(max_ur_raw, (int, float)) else 0.0
            if min_rms is None or max_rms is None or min_rms == 0:
                severity = "UNKNOWN"
                message = "Insufficient gradient data."
            elif min_rms < 1e-5 or min_ur < 1e-4:
                # Vanishing: RMS < 1e-5 OR update ratio < 1e-4 (matches slide 5F).
                worst_layer = (
                    stats.sort("rms").row(0, named=True)["layer"]
                    if min_rms < 1e-5
                    else stats.sort("ur").row(0, named=True)["layer"]
                )
                severity = "CRITICAL"
                message = (
                    f"Vanishing gradients at '{worst_layer}' — "
                    f"min RMS = {min_rms:.2e}, min update_ratio = {min_ur:.2e}. "
                    "Fix: verify pre-norm layout (LayerNorm/RMSNorm before block), "
                    "add residual connections, switch to GELU/SiLU, or use Kaiming init."
                )
            elif max_rms > 1e-2 or max_ur > 0.1:
                # Exploding: RMS > 1e-2 OR update ratio > 0.1 (matches slide 5F).
                worst_layer = (
                    stats.sort("rms", descending=True).row(0, named=True)["layer"]
                    if max_rms > 1e-2
                    else stats.sort("ur", descending=True).row(0, named=True)["layer"]
                )
                severity = "CRITICAL"
                message = (
                    f"Exploding gradients at '{worst_layer}' — "
                    f"max RMS = {max_rms:.2e}, max update_ratio = {max_ur:.2e}. "
                    "Fix: add gradient clipping (or adaptive: ZClip/AGC), reduce LR, "
                    "verify initialization (Kaiming for ReLU, Xavier for Tanh)."
                )
            elif max_rms / max(min_rms, 1e-12) > 1e3:
                severity = "WARNING"
                message = (
                    f"Large RMS spread across layers (max/min = "
                    f"{max_rms / min_rms:.1e}). Deep layers may be learning unevenly."
                )
            else:
                severity = "HEALTHY"
                message = (
                    f"Gradient flow OK (RMS range {min_rms:.2e}–{max_rms:.2e}, "
                    f"update_ratio range {min_ur:.2e}–{max_ur:.2e})."
                )
            findings["gradient_flow"] = {"severity": severity, "message": message}
        elif gdf.height:
            # Legacy path for dataframes without the new columns.
            findings["gradient_flow"] = {
                "severity": "UNKNOWN",
                "message": (
                    "grad_rms field missing — re-run with the current library "
                    "version to get scale-invariant diagnosis."
                ),
            }
        else:
            findings["gradient_flow"] = {
                "severity": "UNKNOWN",
                "message": "No gradient tracking enabled — call track_gradients().",
            }

        # 2. Dead neurons / saturation — uses ACTIVATION-TYPE-AWARE
        # inactivity_fraction from _act_log (ReLU: |x|<eps; Tanh: |x|>0.99;
        # Sigmoid: |x|>0.99 or |x|<0.01). This catches near-dead ReLU channels
        # that the legacy exact-zero check misses post-BatchNorm.
        adf = self.activations_df()
        if adf.height and "inactivity_fraction" in adf.columns:
            # Aggregate mean inactivity per layer (averaged across batches).
            agg = (
                adf.filter(pl.col("act_kind") != "other")
                .group_by("layer")
                .agg(
                    pl.col("inactivity_fraction").mean().alias("mean_inactive"),
                    pl.col("act_kind").first().alias("kind"),
                )
                .sort("mean_inactive", descending=True)
            )
            if agg.height:
                worst = agg.row(0, named=True)
                thr = self.dead_neuron_threshold
                if worst["mean_inactive"] > thr:
                    kind = worst["kind"]
                    if kind == "relu":
                        label = "dead neurons"
                        fix = "Switch to GELU/LeakyReLU or re-initialise with Kaiming."
                    elif kind == "tanh":
                        label = "saturated (|x|>0.99)"
                        fix = "Reduce LR, add LayerNorm before, or switch to GELU."
                    elif kind == "sigmoid":
                        label = "saturated (|x|>0.99 or |x|<0.01)"
                        fix = "Reduce LR, add BN/LN, or switch to softmax+CE if classification."
                    else:
                        label = "inactive"
                        fix = "Investigate activation distribution."
                    findings["dead_neurons"] = {
                        "severity": "WARNING",
                        "message": (
                            f"'{worst['layer']}' ({kind}): "
                            f"{worst['mean_inactive']:.0%} {label}. {fix}"
                        ),
                    }
                else:
                    findings["dead_neurons"] = {
                        "severity": "HEALTHY",
                        "message": (
                            f"All {agg.height} activation layers healthy "
                            f"(worst: {worst['layer']} at "
                            f"{worst['mean_inactive']:.0%} inactive, below "
                            f"{thr:.0%} threshold)."
                        ),
                    }
            else:
                findings["dead_neurons"] = {
                    "severity": "UNKNOWN",
                    "message": "No activation layers tracked — call track_activations().",
                }
        else:
            findings["dead_neurons"] = {
                "severity": "UNKNOWN",
                "message": "No activation tracking enabled — call track_activations().",
            }

        # 3. Loss trend
        edf = self.epochs_df()
        if edf.height >= 3 and edf["val_loss"].is_not_null().any():
            overfit = self._detect_overfit_epoch()
            train_trend = self._slope(edf["train_loss"].to_list())
            if overfit is not None:
                severity = "WARNING"
                message = (
                    f"Overfitting suspected at epoch {overfit} "
                    "(val loss rising while train loss falls). "
                    "Consider dropout, weight decay, or early stopping."
                )
            elif train_trend > -1e-4 and edf.height >= 5:
                severity = "WARNING"
                message = (
                    f"Underfitting — train loss slope {train_trend:.2e}/epoch. "
                    "Consider a larger model, more epochs, or higher LR."
                )
            else:
                severity = "HEALTHY"
                message = f"Loss converging (train slope {train_trend:.2e}/epoch)."
            findings["loss_trend"] = {"severity": severity, "message": message}
        else:
            findings["loss_trend"] = {
                "severity": "UNKNOWN",
                "message": "Need at least 3 epochs with val_loss for trend analysis.",
            }

        # Human-readable summary
        print("\n" + "═" * 64)
        print("  DL Diagnostics Report — Prescription Pad")
        print("═" * 64)
        icons = {
            "HEALTHY": "✓",
            "WARNING": "!",
            "CRITICAL": "X",
            "UNKNOWN": "?",
        }
        titles = {
            "gradient_flow": "Gradient flow",
            "dead_neurons": "Dead neurons ",
            "loss_trend": "Loss trend   ",
        }
        for key, label in titles.items():
            f = findings[key]
            print(
                f"  [{icons[f['severity']]}] {label} ({f['severity']}): {f['message']}"
            )
        print("═" * 64 + "\n")
        return findings

    # ── Grad-CAM ──────────────────────────────────────────────────────────

    def grad_cam(
        self,
        input_tensor: torch.Tensor,
        target_class: int,
        layer_name: str,
    ) -> torch.Tensor:
        """Compute a Grad-CAM heatmap for ``target_class`` from ``layer_name``.

        Args:
            input_tensor: Input batch ``(N, C, H, W)`` or ``(N, C, L)``.
            target_class: Target class index to explain.
            layer_name: ``model.named_modules()`` key of the conv layer to
                attribute. Usually the last convolutional layer.

        Returns:
            Heatmap tensor with shape matching the spatial dims of the tracked
            layer's output (``(N, H', W')`` for 2D inputs).

        Raises:
            ValueError: If ``layer_name`` is not found in the model.
        """
        target_module: nn.Module | None = None
        for name, module in self.model.named_modules():
            if name == layer_name:
                target_module = module
                break
        if target_module is None:
            raise ValueError(
                f"Layer '{layer_name}' not found in model. "
                f"Available: {[n for n, _ in self.model.named_modules() if n][:10]}..."
            )

        self._gradcam_activation = None
        self._gradcam_gradient = None

        def fwd_hook(_mod: nn.Module, _inp: Any, out: torch.Tensor) -> None:
            self._gradcam_activation = out.detach()

        def bwd_hook(_mod: nn.Module, _gi: Any, go: Any) -> None:
            # go is a tuple — first element is d(output)/d(module-output)
            self._gradcam_gradient = go[0].detach()

        h1 = target_module.register_forward_hook(fwd_hook)
        h2 = target_module.register_full_backward_hook(bwd_hook)
        self._handles.grad_cam.extend([h1, h2])

        # Preserve caller's train/eval state — critical for mid-training use.
        was_training = self.model.training

        try:
            self.model.eval()
            inp = input_tensor.to(self.device)
            logits = self.model(inp)
            if logits.ndim != 2:
                raise ValueError(
                    f"grad_cam expects classification logits (N, C); got {logits.shape}"
                )
            self.model.zero_grad(set_to_none=True)
            one_hot = torch.zeros_like(logits)
            one_hot[:, target_class] = 1.0
            logits.backward(gradient=one_hot, retain_graph=False)

            if self._gradcam_activation is None or self._gradcam_gradient is None:
                raise RuntimeError(
                    "Grad-CAM hooks did not fire — layer may be unreachable "
                    "from the forward path."
                )
            activations = self._gradcam_activation  # (N, K, ...)
            gradients = self._gradcam_gradient  # (N, K, ...)
            # Global-average-pool gradients over spatial dims to get per-channel weights.
            spatial_dims = tuple(range(2, gradients.ndim))
            weights = gradients.mean(dim=spatial_dims, keepdim=True)  # (N, K, 1, ...)
            cam = (weights * activations).sum(dim=1)  # (N, ...)
            cam = torch.relu(cam)
            # Normalise per-sample to [0, 1].
            flat = cam.flatten(start_dim=1)
            mins = flat.min(dim=1, keepdim=True).values
            maxs = flat.max(dim=1, keepdim=True).values
            denom = (maxs - mins).clamp(min=1e-8)
            flat = (flat - mins) / denom
            cam = flat.view_as(cam)
            return cam.cpu()
        finally:
            # Restore caller's train/eval state BEFORE removing hooks, so
            # downstream errors in hook cleanup don't leave model in eval mode.
            if was_training:
                self.model.train()
            h1.remove()
            h2.remove()
            # Remove from bookkeeping too so detach() stays idempotent.
            self._handles.grad_cam = [
                h for h in self._handles.grad_cam if h is not h1 and h is not h2
            ]
            self._gradcam_activation = None
            self._gradcam_gradient = None

    # ── LR range test (static) ────────────────────────────────────────────

    @staticmethod
    def lr_range_test(
        model: nn.Module,
        dataloader: DataLoader,
        *,
        loss_fn: nn.Module | None = None,
        optimizer_cls: type[torch.optim.Optimizer] = torch.optim.SGD,
        lr_min: float = 1e-7,
        lr_max: float = 10.0,
        steps: int = 200,
        device: torch.device | None = None,
        batch_adapter: Callable[[Any], tuple[torch.Tensor, torch.Tensor]] | None = None,
        safety_divisor: float = 10.0,
    ) -> dict[str, Any]:
        """Leslie Smith LR range test (with fastai-style safe-LR recipe).

        Trains for ``steps`` batches while exponentially increasing LR from
        ``lr_min`` to ``lr_max``. Smooths the loss curve with EMA (beta=0.98)
        before finding the steepest-descent point — this matches fastai's
        ``lr_find()`` and avoids picking a single lucky batch in the tail.

        **Critical**: returns BOTH ``min_loss_lr`` (steepest descent) AND
        ``safe_lr = min_loss_lr / safety_divisor`` (default 10). Use ``safe_lr``
        in your optimizer — ``min_loss_lr`` is the edge of instability.

        The model's weights are saved before the test and **restored** on exit
        (via state_dict deepcopy), so calling this does not corrupt your model.

        Args:
            model: The model to probe. Weights are restored after return.
            dataloader: Any DataLoader. By default yields ``(inputs, targets)``
                tuples; pass ``batch_adapter`` for HF/SSL batch formats.
            loss_fn: Loss function (REQUIRED — no silent default).
            optimizer_cls: Optimizer class (default SGD).
            lr_min, lr_max, steps: Sweep configuration.
            device: Override compute device (default: model's current device).
            batch_adapter: Callable ``batch -> (x, y)``. Default handles
                tuple/list; pass a lambda for dict-shaped batches (e.g. HF).
            safety_divisor: Divide steepest-descent LR by this to get safe LR
                (fastai default: 10).

        Returns:
            ``{"safe_lr": float,              # use this in your optimizer
                "min_loss_lr": float,          # steepest descent (edge of instability)
                "divergence_lr": float,        # where smoothed loss > 4× min
                "lrs": [...], "losses": [...], "losses_smooth": [...],
                "figure": go.Figure}``
        """
        if steps < 2:
            raise ValueError("steps must be >= 2")
        if not 0 < lr_min < lr_max:
            raise ValueError("require 0 < lr_min < lr_max")
        if loss_fn is None:
            raise ValueError(
                "loss_fn is required — no silent default. "
                "Pass nn.CrossEntropyLoss() for classification or nn.MSELoss() for regression."
            )

        import copy as _copy

        # Device: honor model's current device unless overridden.
        dev = device or next(model.parameters()).device
        if device is not None:
            model = model.to(dev)

        # Save weights for restoration.
        saved_state = _copy.deepcopy(model.state_dict())

        # Default batch adapter handles tuple/list; raises loudly on dicts.
        def _default_adapter(batch: Any) -> tuple[torch.Tensor, torch.Tensor]:
            if isinstance(batch, (tuple, list)) and len(batch) >= 2:
                return batch[0], batch[1]
            raise ValueError(
                "lr_range_test got a non-(x,y) batch. Pass batch_adapter=... "
                "for HuggingFace-style dict batches or SSL single-tensor batches."
            )

        adapter = batch_adapter or _default_adapter

        try:
            optimizer = optimizer_cls(model.parameters(), lr=lr_min)
            gamma = (lr_max / lr_min) ** (1.0 / (steps - 1))
            lrs: list[float] = []
            losses: list[float] = []
            running_min = float("inf")  # O(1) running min, not O(n)
            model.train()
            data_iter = iter(dataloader)
            for step in range(steps):
                try:
                    batch = next(data_iter)
                except StopIteration:
                    data_iter = iter(dataloader)
                    batch = next(data_iter)
                x, y = adapter(batch)
                x, y = x.to(dev), y.to(dev)
                for pg in optimizer.param_groups:
                    pg["lr"] = lr_min * (gamma**step)
                optimizer.zero_grad(set_to_none=True)
                logits = model(x)
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()
                cur_lr = optimizer.param_groups[0]["lr"]
                cur_loss = float(loss.item())
                lrs.append(cur_lr)
                losses.append(cur_loss)
                if cur_loss < running_min:
                    running_min = cur_loss
                if not math.isfinite(cur_loss) or cur_loss > 10 * running_min:
                    logger.info(
                        "dldiagnostics.lr_range_test.diverged",
                        extra={"step": step, "lr": cur_lr, "loss": cur_loss},
                    )
                    break
        finally:
            # Always restore weights — no silent corruption.
            model.load_state_dict(saved_state)

        # EMA-smoothed losses (fastai convention, beta=0.98).
        beta = 0.98
        losses_smooth: list[float] = []
        ema = 0.0
        for i, ell in enumerate(losses):
            ema = beta * ema + (1 - beta) * ell
            # Bias-correct the EMA.
            losses_smooth.append(ema / (1 - beta ** (i + 1)))

        # min_loss_lr = LR at the steepest-descent point of SMOOTHED loss.
        min_loss_lr = lr_min
        divergence_lr = lr_max
        if len(losses_smooth) >= 3:
            dl = np.diff(np.array(losses_smooth))
            idx = int(np.argmin(dl))
            min_loss_lr = lrs[idx]
            # Divergence = first point where smoothed loss > 4× its minimum.
            min_smooth = min(losses_smooth)
            for i, s in enumerate(losses_smooth):
                if s > 4 * min_smooth and i > idx:
                    divergence_lr = lrs[i]
                    break

        safe_lr = min_loss_lr / safety_divisor

        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=lrs,
                y=losses,
                mode="lines",
                name="raw loss",
                line=dict(color="lightgray"),
            )
        )
        fig.add_trace(
            go.Scatter(
                x=lrs,
                y=losses_smooth,
                mode="lines",
                name="smoothed",
                line=dict(color="#0D9488", width=2),
            )
        )
        fig.add_vline(
            x=safe_lr,
            line=dict(color="#22C55E", dash="dash"),
            annotation_text=f"safe_lr = {safe_lr:.2e}",
        )
        fig.add_vline(
            x=min_loss_lr,
            line=dict(color="#F59E0B", dash="dot"),
            annotation_text=f"min_loss_lr = {min_loss_lr:.2e}",
        )
        fig.update_layout(
            title="LR Range Test (smoothed) — use safe_lr, not min_loss_lr",
            xaxis_title="learning rate",
            yaxis_title="loss",
            xaxis_type="log",
            template="plotly_white",
        )
        logger.info(
            "dldiagnostics.lr_range_test.ok",
            extra={
                "safe_lr": safe_lr,
                "min_loss_lr": min_loss_lr,
                "divergence_lr": divergence_lr,
                "steps_run": len(lrs),
            },
        )
        return {
            "safe_lr": safe_lr,
            "min_loss_lr": min_loss_lr,
            "divergence_lr": divergence_lr,
            "suggested_lr": safe_lr,  # backwards-compat alias
            "lrs": lrs,
            "losses": losses,
            "losses_smooth": losses_smooth,
            "figure": fig,
        }

    # ── Hook factories (internal) ─────────────────────────────────────────

    def _make_grad_hook(self, name: str):
        """Scale-invariant gradient tracking.

        Records three metrics per layer per batch:
          - grad_norm: raw L2 norm (preserved for backwards compatibility)
          - grad_rms: per-element RMS = ‖∇‖ / sqrt(numel) — scale-invariant,
            comparable across layers of any size. This is what LLM training
            dashboards (W&B, TensorBoard) actually display.
          - update_ratio: ‖∇W‖ / ‖W‖ — the "effective LR" per layer.

        Casts to fp32 before reduction so BF16/FP16 gradients don't silently
        produce inf/NaN.
        """
        # Look up the parameter tensor for update-ratio computation.
        try:
            param = dict(self.model.named_parameters())[name]
        except KeyError:
            param = None

        def _hook(grad: torch.Tensor) -> None:
            try:
                # Handle sparse gradients (e.g. nn.Embedding(sparse=True)).
                g = grad.coalesce().values() if grad.is_sparse else grad
                # Cast to fp32 to avoid BF16/FP16 inf.
                g_f32 = g.detach().float()
                l2 = float(g_f32.norm(p=2).item())
                numel = max(g_f32.numel(), 1)
                rms = l2 / (numel**0.5)
                # Update ratio — use detached param weight norm.
                if param is not None:
                    p_norm = float(param.detach().float().norm(p=2).item())
                    upd_ratio = l2 / max(p_norm, 1e-12)
                else:
                    upd_ratio = 0.0
            except Exception as exc:  # pragma: no cover - defensive
                logger.warning(
                    "dldiagnostics.grad_hook.error",
                    extra={"layer": name, "error": str(exc)},
                )
                return
            self._grad_log.append(
                {
                    "batch": self._batch_idx,
                    "layer": name,
                    "grad_norm": l2,  # preserved for compatibility
                    "grad_rms": rms,  # scale-invariant
                    "update_ratio": upd_ratio,  # ‖∇W‖ / ‖W‖
                }
            )

        return _hook

    def _make_act_hook(self, name: str):
        """Activation statistics. Casts to fp32 to avoid BF16/FP16 overflow.

        The `inactivity_fraction` field measures activation-type-appropriate
        pathology:
          - ReLU / GELU / SiLU: fraction with |x| < 1e-6 (dead neurons)
          - Tanh: fraction with |x| > 0.99 (saturated)
          - Sigmoid: fraction with |x| > 0.99 OR |x| < 0.01 (saturated)
          - Others: 0 (no pathology definition)

        The legacy `dead_fraction` field (exact-zero count) is preserved for
        backwards compatibility but is only meaningful for ReLU.
        """
        # Determine activation type from module class name for dispatching.
        act_kind = self._classify_activation_module(name)

        def _hook(_module: nn.Module, _inp: Any, output: torch.Tensor) -> None:
            if not isinstance(output, torch.Tensor) or output.numel() == 0:
                return
            # Cast to fp32 for numerical safety (BF16/FP16 overflow on .std()).
            out = output.detach().float()
            try:
                mean = float(out.mean().item())
                std = float(out.std().item()) if out.numel() > 1 else 0.0
                mn = float(out.min().item())
                mx = float(out.max().item())
                legacy_dead = float((out == 0).float().mean().item())
                # Activation-aware inactivity/saturation metric.
                if act_kind == "relu":
                    inactivity = float((out.abs() < 1e-6).float().mean().item())
                elif act_kind == "tanh":
                    inactivity = float((out.abs() > 0.99).float().mean().item())
                elif act_kind == "sigmoid":
                    inactivity = float(
                        ((out > 0.99) | (out < 0.01)).float().mean().item()
                    )
                else:
                    inactivity = 0.0
            except RuntimeError:
                # Non-numeric tensor (e.g. mixed dtype). Skip silently.
                return
            # Guard against NaN/inf leaking into logs.
            for val_name, val in (("mean", mean), ("std", std)):
                if val != val or val in (float("inf"), float("-inf")):
                    logger.warning(
                        "dldiagnostics.act_hook.nonfinite",
                        extra={"layer": name, "field": val_name},
                    )
                    return
            self._act_log.append(
                {
                    "batch": self._batch_idx,
                    "layer": name,
                    "act_kind": act_kind,
                    "mean": mean,
                    "std": std,
                    "min": mn,
                    "max": mx,
                    "dead_fraction": legacy_dead,
                    "inactivity_fraction": inactivity,
                }
            )

        return _hook

    def _classify_activation_module(self, name: str) -> str:
        """Return one of 'relu', 'tanh', 'sigmoid', 'other' for a module name."""
        try:
            mod = dict(self.model.named_modules())[name]
        except KeyError:
            return "other"
        cls = type(mod).__name__.lower()
        if any(k in cls for k in ("relu", "gelu", "silu", "swish", "elu")):
            return "relu"
        if "tanh" in cls:
            return "tanh"
        if "sigmoid" in cls:
            return "sigmoid"
        return "other"

    def _make_dead_hook(self, name: str):
        def _hook(_module: nn.Module, _inp: Any, output: torch.Tensor) -> None:
            if not isinstance(output, torch.Tensor) or output.numel() == 0:
                return
            out = output.detach()
            # Collapse all non-channel dims. Convention: channel dim = 1 for
            # Conv outputs (N, C, ...); for Linear, output is (N, F) — here
            # we treat dim 1 as "neurons" which matches both shapes.
            if out.ndim < 2:
                return
            reduce_dims = tuple(d for d in range(out.ndim) if d != 1)
            fired = (out != 0).any(dim=reduce_dims).float().cpu()
            if name not in self._firing_counts:
                self._firing_counts[name] = torch.zeros_like(fired)
                self._firing_samples[name] = 0
            self._firing_counts[name] += fired
            self._firing_samples[name] += 1
            # Keep memory bounded by decaying old counts when window exceeded.
            if self._firing_samples[name] > self.window:
                scale = self.window / self._firing_samples[name]
                self._firing_counts[name] = self._firing_counts[name] * scale
                self._firing_samples[name] = self.window

        return _hook

    # ── Internal analysis helpers ─────────────────────────────────────────

    @staticmethod
    def _slope(series: list[float]) -> float:
        """Least-squares slope of a 1D series vs its index (ignores NaN)."""
        xs = np.arange(len(series), dtype=float)
        ys = np.asarray(series, dtype=float)
        mask = np.isfinite(ys)
        if mask.sum() < 2:
            return 0.0
        xs, ys = xs[mask], ys[mask]
        return float(np.polyfit(xs, ys, 1)[0])

    def _detect_overfit_epoch(self) -> int | None:
        """Return epoch index where val loss starts rising while train falls."""
        edf = self.epochs_df()
        if edf.height < 3:
            return None
        train = edf["train_loss"].to_list()
        val = edf["val_loss"].to_list()
        for i in range(2, len(val)):
            v_now, v_prev = val[i], val[i - 1]
            t_now, t_prev = train[i], train[i - 1]
            if any(
                x is None or not math.isfinite(x)
                for x in (v_now, v_prev, t_now, t_prev)
            ):
                continue
            if v_now > v_prev and t_now < t_prev:
                # Confirm trend persists one step back if available.
                return i
        return None


# ════════════════════════════════════════════════════════════════════════
# Standalone diagnostic checkpoint — for use AFTER a training loop
# ════════════════════════════════════════════════════════════════════════


def run_diagnostic_checkpoint(
    model: nn.Module,
    dataloader: DataLoader,
    loss_fn: Callable[..., Any],
    *,
    title: str = "Model",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    batch_adapter: Callable[[Any], tuple[torch.Tensor, ...]] | None = None,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Run a short instrumented diagnostic pass on a TRAINED model.

    Use this between the Train and Visualise phases of an exercise. It
    attaches the four diagnostic instruments (gradients, activations,
    dead-neurons, scalar history) to the model, runs ``n_batches`` of
    forward-backward passes to populate the history, replays any
    epoch-level losses you collected during real training, and prints the
    Prescription Pad with auto-diagnosis.

    The model's weights are NOT updated — gradients are computed but no
    optimizer step is taken. The model's train/eval state is preserved.

    Args:
        model: A trained ``nn.Module``.
        dataloader: Any DataLoader whose batches the loss function accepts.
        loss_fn: Callable. The default contract is
            ``loss_fn(model, batch) -> (scalar_loss, extras_dict)`` matching
            the M5 exercise convention. Pass ``batch_adapter`` to wrap a
            different signature.
        title: Human-readable name printed in the dashboard title.
        n_batches: How many batches to instrument (default 8 — enough for
            a stable mean per layer without slowing the exercise down).
        train_losses: Optional list of per-epoch train losses captured
            during the actual training run. Replayed into the dashboard's
            stethoscope view so students see the real loss trajectory, not
            just the diagnostic-pass losses.
        val_losses: Optional list of per-epoch validation losses, same
            length as ``train_losses``.
        show: If ``True``, calls ``.show()`` on the dashboard figure.
        batch_adapter: Optional ``batch -> (loss_fn_args...)`` translator
            for non-standard batch shapes.

    Returns:
        ``(diag, findings)`` so the caller can inspect the DataFrames and
        the Prescription Pad findings dict.
    """
    if not isinstance(model, nn.Module):
        raise TypeError("run_diagnostic_checkpoint requires an nn.Module")
    if n_batches < 1:
        raise ValueError("n_batches must be >= 1")

    diag = DLDiagnostics(model)
    diag.track_gradients().track_activations().track_dead_neurons()

    # Replay real training history into the dashboard if provided.
    if train_losses or val_losses:
        n_epochs = max(len(train_losses or []), len(val_losses or []))
        for i in range(n_epochs):
            tl = (
                float(train_losses[i])
                if train_losses and i < len(train_losses)
                else None
            )
            vl = float(val_losses[i]) if val_losses and i < len(val_losses) else None
            # Synthesise one batch entry per epoch so the per-batch stethoscope
            # has data to plot — students still see the real epoch losses.
            if tl is not None:
                diag.record_batch(loss=tl)
            diag.record_epoch(train_loss=tl, val_loss=vl)

    was_training = model.training
    try:
        model.train()  # Enable gradients + activation hooks.
        seen = 0
        for batch in dataloader:
            if seen >= n_batches:
                break
            try:
                if batch_adapter is not None:
                    args = batch_adapter(batch)
                    loss_out = loss_fn(model, *args)
                else:
                    loss_out = loss_fn(model, batch)
                # Convention: M5 loss_fn returns (loss, extras_dict). Allow
                # either a bare tensor or a tuple.
                if isinstance(loss_out, tuple):
                    loss = loss_out[0]
                else:
                    loss = loss_out
                if not isinstance(loss, torch.Tensor):
                    raise TypeError(
                        f"loss_fn returned {type(loss).__name__}; expected Tensor"
                    )
                model.zero_grad(set_to_none=True)
                loss.backward()
                # NOTE: no optimizer.step() — diagnostic pass is read-only.
                diag.record_batch(loss=float(loss.item()))
            except Exception as exc:  # pragma: no cover - student loop variability
                logger.warning(
                    "dldiagnostics.checkpoint.batch_skipped",
                    extra={"batch_idx": seen, "error": str(exc)},
                )
            seen += 1
    finally:
        if not was_training:
            model.eval()

    # Render the dashboard and the Prescription Pad.
    fig = diag.plot_training_dashboard()
    fig.update_layout(title=f"{title} — Diagnostic Dashboard (Vital Signs)")
    if show:
        try:
            fig.show()
        except Exception as exc:  # pragma: no cover - no display in CI
            logger.info(
                "dldiagnostics.checkpoint.show_skipped",
                extra={"error": str(exc)},
            )

    findings = diag.report()
    return diag, findings


def diagnose_classifier(
    model: nn.Module,
    dataloader: DataLoader,
    *,
    title: str = "Classifier",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    forward_returns_tuple: bool = False,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Convenience wrapper for ``(x, y)`` cross-entropy classifiers.

    Equivalent to :func:`run_diagnostic_checkpoint` with a built-in
    ``loss_fn`` that calls ``F.cross_entropy(model(x), y)``. Use this for
    CNN, transformer, and transfer-learning exercises.

    Args:
        model: Trained classifier ``nn.Module``.
        dataloader: Yields ``(x, y)`` tuples; ``y`` is class indices.
        title: Title for the dashboard.
        n_batches: Batches to instrument.
        train_losses, val_losses: Optional epoch histories to replay.
        show: Show the figure inline.
        forward_returns_tuple: If ``True``, ``model(x)`` returns
            ``(logits, ...)`` (e.g. attention models) — only the first
            element is used as logits.

    Returns:
        ``(diag, findings)``
    """
    import torch.nn.functional as F  # local — torch already imported

    def _cls_loss(m: nn.Module, batch: Any) -> torch.Tensor:
        x, y = batch[0], batch[1]
        out = m(x)
        logits = out[0] if forward_returns_tuple else out
        return F.cross_entropy(logits, y)

    return run_diagnostic_checkpoint(
        model,
        dataloader,
        _cls_loss,
        title=title,
        n_batches=n_batches,
        train_losses=train_losses,
        val_losses=val_losses,
        show=show,
    )


def diagnose_regressor(
    model: nn.Module,
    dataloader: DataLoader,
    *,
    title: str = "Regressor",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    forward_returns_tuple: bool = False,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Convenience wrapper for ``(x, y)`` MSE regressors (RNN exercises).

    Equivalent to :func:`run_diagnostic_checkpoint` with a built-in
    ``loss_fn`` that calls ``F.mse_loss(model(x), y)``.

    Args:
        forward_returns_tuple: Set ``True`` for attention models that
            return ``(predictions, attn_weights)``.
    """
    import torch.nn.functional as F

    def _reg_loss(m: nn.Module, batch: Any) -> torch.Tensor:
        x, y = batch[0], batch[1]
        out = m(x)
        pred = out[0] if forward_returns_tuple else out
        return F.mse_loss(pred, y)

    return run_diagnostic_checkpoint(
        model,
        dataloader,
        _reg_loss,
        title=title,
        n_batches=n_batches,
        train_losses=train_losses,
        val_losses=val_losses,
        show=show,
    )


__all__ = [
    "DLDiagnostics",
    "run_diagnostic_checkpoint",
    "diagnose_classifier",
    "diagnose_regressor",
]


# ── shared/mlfp05/ex_2.py ──
"""
Shared utilities for MLFP05 Exercise 2 — CNNs on CIFAR-10.

Common infrastructure used by all technique files:
  - CIFAR-10 data loading and normalisation
  - Device detection and precision settings
  - ExperimentTracker / ModelRegistry setup
  - LightningModule wrapper for training
  - Visualisation helpers
"""

import asyncio
import pickle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import pytorch_lightning as pl
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ModelVisualizer
from kailash_ml.bridge.onnx_bridge import OnnxBridge
from kailash_ml.engines.experiment_tracker import ExperimentTracker
from kailash_ml.engines.inference_server import InferenceServer
from kailash_ml.engines.model_registry import ModelRegistry
from kailash_ml.types import MetricSpec

# ── Environment & Reproducibility ────────────────────────────────────
setup_environment()
torch.manual_seed(42)
np.random.seed(42)
pl.seed_everything(42, workers=True)

DEVICE = get_device()
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cifar10"
ARTIFACT_DIR = Path(__file__).resolve().parent

N_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 8

# ── CIFAR-10 class labels (used for visualisation and apply sections) ─
CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Per-channel normalisation statistics (CIFAR-10 population)
CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)


# ═══════════════════════════════════════════════════════════════════════
# Data Loading
# ═══════════════════════════════════════════════════════════════════════
def load_cifar10() -> tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    DataLoader,
    DataLoader,
]:
    """Load and normalise the full CIFAR-10 dataset.

    Returns:
        X_train, y_train, X_val, y_val, train_loader, val_loader
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    train_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    val_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_train = torch.stack([train_set[i][0] for i in range(len(train_set))])
    y_train = torch.tensor(
        [train_set[i][1] for i in range(len(train_set))],
        dtype=torch.long,
    )
    X_val = torch.stack([val_set[i][0] for i in range(len(val_set))])
    y_val = torch.tensor(
        [val_set[i][1] for i in range(len(val_set))],
        dtype=torch.long,
    )

    # Per-channel normalisation
    X_train = (X_train - CIFAR_MEAN) / CIFAR_STD
    X_val = (X_val - CIFAR_MEAN) / CIFAR_STD

    train_ds = TensorDataset(X_train, y_train)
    val_ds = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    print(
        f"CIFAR-10: train {tuple(X_train.shape)}, val {tuple(X_val.shape)}, "
        f"classes={N_CLASSES}: {CLASS_NAMES}"
    )
    return X_train, y_train, X_val, y_val, train_loader, val_loader


# ═══════════════════════════════════════════════════════════════════════
# Kailash Engine Setup
# ═══════════════════════════════════════════════════════════════════════
async def setup_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Initialise ExperimentTracker and ModelRegistry.

    Returns:
        conn, tracker, experiment_name, registry (or None), has_registry
    """
    conn = ConnectionManager(f"sqlite:///{db_name}")
    await conn.initialize()

    tracker = ExperimentTracker(conn)
    exp_name = await tracker.create_experiment(
        name="m5_cnns",
        description="CNN architectures on full CIFAR-10 (50K images)",
    )

    try:
        registry = ModelRegistry(conn)
        has_registry = True
    except Exception as e:
        registry = None
        has_registry = False
        print(f"  Note: ModelRegistry setup skipped ({e})")

    return conn, tracker, exp_name, registry, has_registry


def init_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Sync wrapper for setup_engines."""
    return asyncio.run(setup_engines(db_name))


# ═══════════════════════════════════════════════════════════════════════
# Lightning Training Infrastructure
# ═══════════════════════════════════════════════════════════════════════
class LitCNN(pl.LightningModule):
    """Lightning wrapper for any nn.Module classifier.

    Tracks per-epoch training loss and validation accuracy for later
    logging to ExperimentTracker and visualisation with ModelVisualizer.
    """

    def __init__(self, model: nn.Module, lr: float = 1e-3):
        super().__init__()
        self.model = model
        self.lr = lr
        self.train_losses: list[float] = []
        self.val_accs: list[float] = []
        self._batch_losses: list[float] = []
        self._val_correct = 0
        self._val_total = 0

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self.model(x)
        loss = F.cross_entropy(logits, y)
        self._batch_losses.append(loss.item())
        return loss

    def on_train_epoch_end(self):
        if self._batch_losses:
            self.train_losses.append(float(np.mean(self._batch_losses)))
            self._batch_losses = []

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self.model(x)
        self._val_correct += int((logits.argmax(dim=-1) == y).sum().item())
        self._val_total += int(y.size(0))

    def on_validation_epoch_end(self):
        if self._val_total > 0:
            self.val_accs.append(self._val_correct / self._val_total)
            self._val_correct = 0
            self._val_total = 0

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=self.lr)


def get_precision_setting() -> str:
    """Detect whether mixed-precision training is beneficial."""
    if torch.cuda.is_available():
        cap = torch.cuda.get_device_capability()
        if cap[0] >= 7:
            print("  GPU with Tensor Cores detected -- using 16-mixed precision")
            return "16-mixed"
        print("  Older GPU detected -- using 32-bit precision")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("  Apple MPS detected -- using 32-bit precision (no fp16 compute units)")
    else:
        print("  CPU detected -- using 32-bit precision")
    return "32"


def get_accelerator() -> str:
    """Return the Lightning accelerator string for the current device."""
    if torch.cuda.is_available():
        return "gpu"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


PRECISION = get_precision_setting()
ACCELERATOR = get_accelerator()


async def train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Train a CNN with Lightning and log metrics to ExperimentTracker.

    Uses the ``tracker.run(...)`` async context manager. On normal exit
    the run is marked COMPLETED; on exception it is marked FAILED.
    """
    lit = LitCNN(model, lr=lr)
    trainer = pl.Trainer(
        max_epochs=epochs,
        accelerator=ACCELERATOR,
        precision=PRECISION,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        enable_checkpointing=False,
    )

    async with tracker.run(experiment_name=exp_name, run_name=name) as ctx:
        await ctx.log_params(
            {
                "architecture": name,
                "lr": str(lr),
                "epochs": str(epochs),
                "batch_size": str(BATCH_SIZE),
                "dataset_size": str(len(train_loader.dataset)),
                "precision": PRECISION,
                "accelerator": ACCELERATOR,
            }
        )

        trainer.fit(lit, train_loader, val_loader)

        for epoch_idx, loss in enumerate(lit.train_losses):
            await ctx.log_metric("train_loss", loss, step=epoch_idx + 1)
        for epoch_idx, acc in enumerate(lit.val_accs):
            await ctx.log_metric("val_accuracy", acc, step=epoch_idx + 1)

        await ctx.log_metrics(
            {
                "final_train_loss": lit.train_losses[-1],
                "final_val_accuracy": lit.val_accs[-1],
            }
        )

    print(
        f"  [{name}] final train loss={lit.train_losses[-1]:.4f}  "
        f"val acc={lit.val_accs[-1]:.3f}"
    )
    return lit.train_losses, lit.val_accs


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        train_model_async(
            model,
            name,
            tracker,
            exp_name,
            train_loader,
            val_loader,
            lr,
            epochs,
        )
    )


# ═══════════════════════════════════════════════════════════════════════
# Model Registration
# ═══════════════════════════════════════════════════════════════════════
async def register_model_async(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Register a trained model with metrics in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="val_accuracy", value=final_acc),
            MetricSpec(name="epochs", value=float(epochs)),
            MetricSpec(name="batch_size", value=float(BATCH_SIZE)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, acc={final_acc:.3f}")
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Sync wrapper for register_model_async."""
    return asyncio.run(
        register_model_async(registry, name, model, final_loss, final_acc, epochs)
    )


# ═══════════════════════════════════════════════════════════════════════
# Visualisation Helpers
# ═══════════════════════════════════════════════════════════════════════
def create_visualizer() -> ModelVisualizer:
    """Return a configured ModelVisualizer instance."""
    return ModelVisualizer()


def save_training_plots(
    viz: ModelVisualizer,
    metrics: dict[str, list[float]],
    output_path: str | Path,
    x_label: str = "Epoch",
    y_label: str = "Value",
) -> None:
    """Save a training history plot to HTML."""
    fig = viz.training_history(
        metrics=metrics,
        x_label=x_label,
        y_label=y_label,
    )
    fig.write_html(str(output_path))
    print(f"  Saved plot: {output_path}")


def count_parameters(model: nn.Module) -> int:
    """Count total trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters())


def denormalise_cifar(img_tensor: torch.Tensor) -> torch.Tensor:
    """Reverse CIFAR-10 normalisation for display.

    Args:
        img_tensor: shape (C, H, W) or (B, C, H, W), normalised

    Returns:
        Tensor with pixel values clipped to [0, 1]
    """
    if img_tensor.dim() == 3:
        mean = CIFAR_MEAN.squeeze(0)
        std = CIFAR_STD.squeeze(0)
    else:
        mean = CIFAR_MEAN
        std = CIFAR_STD
    return (img_tensor * std + mean).clamp(0, 1)




# ════════════════════════════════════════════════════════════════════════
# MLFP05 Exercise 2.4 — Hyperparameter Study: LR and Data Augmentation
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this file, you will be able to:
#   - Explain WHY hyperparameters matter and the law of diminishing
#     returns in terms a non-technical manager can understand
#   - Run a systematic learning rate sweep (3 values) tracked with
#     ExperimentTracker
#   - Compare data augmentation strategies (none vs horizontal flip +
#     random crop) and measure their impact on generalisation
#   - Visualise the accuracy-vs-compute tradeoff (more epochs and more
#     augmentation have decreasing marginal returns)
#   - Answer the engineering manager's question: "How much compute
#     budget should we allocate?"
#
# PREREQUISITES: M5/ex_2/01_simple_cnn.py, 02_resnet_se.py, and
#   03_production_pipeline.py (CNN architectures, training, ONNX)
# ESTIMATED TIME: ~30 min
#
# DATASET: CIFAR-10 — same 50K training images. For the augmentation
#   comparison, we apply transforms on-the-fly during training.
#
# PHASES:
#   1. THEORY  — Why hyperparameters matter, diminishing returns
#   2. BUILD   — ResNetSE (same architecture, different hyperparameters)
#   3. TRAIN   — Learning rate sweep + augmentation comparison
#   4. VISUALISE — Accuracy-vs-cost tradeoff curves
#   5. APPLY   — "How much compute budget?" for an engineering manager
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

import pytorch_lightning as pl
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, TensorDataset



## PHASE 1 — THEORY: Why Hyperparameters Matter


Three dials to tune:
    1. Learning rate: how big each update step is
    2. Data augmentation: synthetic variety to prevent memorisation
    3. Training duration: how many passes through the data

  The law of diminishing returns means there's an OPTIMAL budget --
  not minimum, not maximum, but the point where each additional dollar
  buys the least improvement.


In [ ]:
# ANALOGY FOR NON-TECHNICAL STAKEHOLDERS:
#
# Think of training a neural network like tuning a radio. The MODEL
# ARCHITECTURE is the radio itself — SimpleCNN vs ResNetSE. But the
# HYPERPARAMETERS are the dials you turn:
#
#   - LEARNING RATE: How far you turn the dial each time. Too far
#     (lr=0.1) and you overshoot the station. Too little (lr=0.00001)
#     and you spend an hour finding it. Just right (lr=0.001) and you
#     converge quickly.
#
#   - DATA AUGMENTATION: Imagine listening to the same song through
#     different speakers, at different volumes, with different EQ
#     settings. You learn to recognise the SONG, not the speaker.
#     Data augmentation (flipping, cropping, colour jitter) teaches
#     the model to recognise the OBJECT, not the specific photo angle.
#
#   - EPOCHS (training duration): How many times you listen to your
#     training playlist. First listen: you catch the chorus. Fifth
#     listen: you know the verses. Twentieth listen: you're not
#     learning anything new, just wasting electricity.
#
# THE LAW OF DIMINISHING RETURNS:
#   Going from 5 to 10 epochs might improve accuracy by 5%.
#   Going from 10 to 20 epochs might improve by 2%.
#   Going from 20 to 40 epochs might improve by 0.5%.
#   Each doubling of compute buys less and less improvement.
#
#   This is why the engineering manager's question — "how much compute
#   budget?" — has a concrete, data-driven answer: run the sweep, find
#   the knee of the curve, and spend your budget THERE.

print("=" * 70)
print("  PHASE 1 — THEORY: Hyperparameters and Diminishing Returns")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: ResNetSE for Hyperparameter Experiments


Residual block for hyperparameter experiments.


Squeeze-and-Excitation block for hyperparameter experiments.


ResNet+SE for hyperparameter experiments.


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: ResNetSE (same architecture, different settings)")
print("=" * 70)


class ResBlock(nn.Module):

    def __init__(self, channels: int):
        super().__init__()
        # TODO: Two Conv2d(channels, channels, 3, padding=1) with BatchNorm2d
        self.conv1 = ____
        self.bn1 = ____
        self.conv2 = ____
        self.bn2 = ____

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: identity + residual forward pass
        identity = x
        out = ____
        out = ____
        return ____


class SEBlock(nn.Module):

    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        # TODO: SE MLP — Linear->ReLU->Linear->Sigmoid
        self.fc = nn.Sequential(
            ____,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        # TODO: squeeze -> excite -> scale
        s = ____
        w = ____
        return ____


class ResNetSE(nn.Module):

    def __init__(self, n_classes: int = N_CLASSES):
        super().__init__()
        # TODO: stem, block1, se1, block2, pool, fc (same as 02 and 03)
        self.stem = nn.Sequential(
            ____,
        )
        self.block1 = ____
        self.se1 = ____
        self.block2 = ____
        self.pool = ____
        self.fc = ____

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: stem -> block1 -> se1 -> block2 -> pool -> flatten -> fc
        ____


param_count = count_parameters(ResNetSE())
print(f"  ResNetSE: {param_count:,} parameters")
print("  Same architecture across all experiments -- only hyperparameters change")



## PHASE 3 — TRAIN: Learning Rate Sweep + Augmentation Comparison


Run one LR sweep trial, logged under its own tracker run.


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 3 — TRAIN: Systematic Hyperparameter Experiments")
print("=" * 70)

# Load data
X_train, y_train, X_val, y_val, train_loader, val_loader = load_cifar10()
conn, tracker, exp_name, registry, has_registry = init_engines()

# ──────────────────────────────────────────────────────────────────────
# EXPERIMENT A: Learning Rate Sweep
# ──────────────────────────────────────────────────────────────────────
LR_SWEEP = [5e-4, 1e-3, 3e-3]
HP_EPOCHS = 6
lr_results: dict[str, dict] = {}

print(
    f"\n  EXPERIMENT A: Learning Rate Sweep ({len(LR_SWEEP)} values, {HP_EPOCHS} epochs each)"
)
print("  " + "-" * 60)


# TODO: Implement the async LR sweep training function
#   1. Create fresh ResNetSE for each LR
#   2. Wrap in LitCNN(model, lr=lr)
#   3. Create pl.Trainer with HP_EPOCHS, ACCELERATOR, PRECISION, no progress bar
#   4. Use tracker.run() async context to log params and metrics
#   5. Return (train_losses, val_accs)
async def train_lr_sweep_async(lr: float) -> tuple[list[float], list[float]]:
    hp_model = ResNetSE()
    lit = LitCNN(hp_model, lr=lr)
    trainer = pl.Trainer(
        max_epochs=HP_EPOCHS,
        accelerator=ACCELERATOR,
        precision=PRECISION,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        enable_checkpointing=False,
    )

    async with tracker.run(
        experiment_name=exp_name, run_name=f"hp_sweep_lr_{lr}"
    ) as ctx:
        await ctx.log_params(
            {
                "architecture": "ResNetSE",
                "lr": str(lr),
                "epochs": str(HP_EPOCHS),
                "sweep_type": "learning_rate",
                "augmentation": "none",
            }
        )

        # TODO: Fit the trainer then log all epoch metrics
        #   trainer.fit(lit, train_loader, val_loader)
        #   Loop through lit.train_losses and lit.val_accs to log each epoch
        ____

        for epoch_idx, loss in enumerate(lit.train_losses):
            ____
        for epoch_idx, acc in enumerate(lit.val_accs):
            ____

        await ctx.log_metrics(
            {
                "final_train_loss": lit.train_losses[-1],
                "final_val_accuracy": lit.val_accs[-1],
            }
        )

    return lit.train_losses, lit.val_accs


for lr in LR_SWEEP:
    name = f"resnet_se_lr{lr}"
    print(f"\n  Training ResNetSE with lr={lr}...")
    t0 = time.perf_counter()
    sweep_losses, sweep_accs  = await train_lr_sweep_async(lr)
    elapsed = time.perf_counter() - t0
    lr_results[name] = {
        "lr": lr,
        "losses": sweep_losses,
        "accs": sweep_accs,
        "time_sec": elapsed,
    }
    print(
        f"    lr={lr}: final_loss={sweep_losses[-1]:.4f}, "
        f"val_acc={sweep_accs[-1]:.3f}, time={elapsed:.1f}s"
    )

# Print LR comparison table
print(f"\n  {'Learning Rate':>15} {'Final Loss':>12} {'Val Acc':>10} {'Time (s)':>10}")
print("  " + "-" * 50)
for name, result in lr_results.items():
    print(
        f"  {result['lr']:>15.4f} {result['losses'][-1]:>12.4f} "
        f"{result['accs'][-1]:>9.3f} {result['time_sec']:>9.1f}"
    )

best_lr_config = max(lr_results.items(), key=lambda x: x[1]["accs"][-1])
best_lr = best_lr_config[1]["lr"]
best_lr_acc = best_lr_config[1]["accs"][-1]
print(f"\n  Best LR: {best_lr} (val_acc={best_lr_acc:.3f})")



### Checkpoint 1: LR sweep complete


Apply CIFAR-10 normalisation to a batch.


Wraps a torchvision dataset to add CIFAR-10 normalisation.


In [ ]:
assert len(lr_results) == len(
    LR_SWEEP
), f"Should have results for all {len(LR_SWEEP)} learning rates"
for name, result in lr_results.items():
    assert result["accs"][-1] > 0.2, (
        f"{name} val_acc={result['accs'][-1]:.3f} is too low -- even suboptimal "
        "LR should learn basic features from 50K images"
    )
print("\n--- Checkpoint 1 passed --- learning rate sweep complete\n")


# ──────────────────────────────────────────────────────────────────────
# EXPERIMENT B: Data Augmentation Comparison
# ──────────────────────────────────────────────────────────────────────
# Data augmentation creates synthetic training variety:
#   - Horizontal flip: a cat facing left is still a cat facing right
#   - Random crop: the object can appear at any position
#   - These teach the model to recognise the OBJECT, not the specific
#     photo composition
#
# We compare: (1) no augmentation vs (2) flip + random crop

print("  EXPERIMENT B: Data Augmentation Comparison")
print("  " + "-" * 60)

DATA_DIR.mkdir(parents=True, exist_ok=True)

# TODO: Create augmented and non-augmented transforms
#   augmented_transform: T.Compose([T.RandomHorizontalFlip(0.5), T.RandomCrop(32, padding=4), T.ToTensor()])
#   no_aug_transform: T.Compose([T.ToTensor()])
augmented_transform = T.Compose(
    [
        ____,
    ]
)

no_aug_transform = T.Compose(
    [
        ____,
    ]
)

# Load raw CIFAR-10 with transforms
train_set_aug = torchvision.datasets.CIFAR10(
    root=str(DATA_DIR),
    train=True,
    download=True,
    transform=augmented_transform,
)
train_set_noaug = torchvision.datasets.CIFAR10(
    root=str(DATA_DIR),
    train=True,
    download=True,
    transform=no_aug_transform,
)


def normalise_batch(batch_imgs: torch.Tensor) -> torch.Tensor:
    return (batch_imgs - CIFAR_MEAN) / CIFAR_STD


class NormalisingDataset(torch.utils.data.Dataset):

    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        img = (img - CIFAR_MEAN.squeeze(0)) / CIFAR_STD.squeeze(0)
        return img, label


aug_loader = DataLoader(
    NormalisingDataset(train_set_aug),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
noaug_loader = DataLoader(
    NormalisingDataset(train_set_noaug),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

# TODO: Train with best LR from sweep, comparing augmented vs non-augmented
#   Loop over [("no_augmentation", noaug_loader), ("flip_crop", aug_loader)]
#   For each: model = ResNetSE(), train_model(model, name, tracker, ..., lr=best_lr, epochs=HP_EPOCHS)
aug_results: dict[str, dict] = {}

for aug_name, loader in [("no_augmentation", noaug_loader), ("flip_crop", aug_loader)]:
    print(f"\n  Training ResNetSE with {aug_name} (lr={best_lr})...")
    model = ____
    t0 = time.perf_counter()
    losses, accs = ____
    elapsed = time.perf_counter() - t0
    aug_results[aug_name] = {
        "losses": losses,
        "accs": accs,
        "time_sec": elapsed,
        "model": model,
    }
    print(
        f"    {aug_name}: final_loss={losses[-1]:.4f}, "
        f"val_acc={accs[-1]:.3f}, time={elapsed:.1f}s"
    )

# Print augmentation comparison
print(f"\n  {'Augmentation':>20} {'Final Loss':>12} {'Val Acc':>10} {'Time (s)':>10}")
print("  " + "-" * 55)
for name, result in aug_results.items():
    print(
        f"  {name:>20} {result['losses'][-1]:>12.4f} "
        f"{result['accs'][-1]:>9.3f} {result['time_sec']:>9.1f}"
    )

aug_improvement = (
    aug_results["flip_crop"]["accs"][-1] - aug_results["no_augmentation"]["accs"][-1]
)
print(f"\n  Augmentation improvement: {aug_improvement:+.3f} accuracy")



### Checkpoint 2: Augmentation comparison complete


In [ ]:
assert len(aug_results) == 2, "Should have results for both augmentation settings"
print("\n--- Checkpoint 2 passed --- augmentation comparison complete\n")



## PHASE 4 — VISUALISE: Accuracy-vs-Cost Tradeoff Curves


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: The Accuracy-vs-Cost Tradeoff")
print("=" * 70)

viz = create_visualizer()

# TODO: Save training plots for LR sweep and augmentation comparison
#   Use save_training_plots for: lr_sweep_loss, lr_sweep_acc, augmentation_loss, augmentation_acc
____
____
____
____

# TODO: Create the "diminishing returns" chart
#   Left subplot: accuracy vs epoch for all LR configs (line chart with markers)
#   Right subplot: marginal improvement per epoch (bar chart)
fig_roi, axes = plt.subplots(1, 2, figsize=(16, 6))
fig_roi.suptitle(
    "The Diminishing Returns of Training: When to Stop Spending",
    fontsize=14,
)

# Left: Accuracy vs epoch for all LR configs
for name, result in lr_results.items():
    epochs_x = list(range(1, len(result["accs"]) + 1))
    axes[0].plot(epochs_x, result["accs"], "o-", label=f"lr={result['lr']}")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Validation Accuracy")
axes[0].set_title("Accuracy vs Training Duration")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Marginal improvement per epoch
for name, result in lr_results.items():
    accs = result["accs"]
    marginal = [accs[0]] + [accs[i] - accs[i - 1] for i in range(1, len(accs))]
    epochs_x = list(range(1, len(marginal) + 1))
    axes[1].bar(
        [x + 0.2 * (list(lr_results.keys()).index(name) - 1) for x in epochs_x],
        marginal,
        width=0.2,
        alpha=0.7,
        label=f"lr={result['lr']}",
    )
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Marginal Accuracy Gain")
axes[1].set_title("Diminishing Returns: Accuracy Gained Per Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color="black", linewidth=0.5)

plt.tight_layout()
plt.savefig("ex_2_04_diminishing_returns.png", dpi=150, bbox_inches="tight")
plt.close(fig_roi)
print("  Saved: ex_2_04_diminishing_returns.png")
print("  Left: accuracy climbs steeply then flattens (the 'knee')")
print("  Right: marginal gains shrink each epoch (the ROI curve)")

# TODO: Create cost-accuracy tradeoff scatter plot
#   Each point = one experiment, x = training time, y = accuracy
#   Add Pareto frontier connecting non-dominated points
fig_cost, ax = plt.subplots(1, 1, figsize=(10, 6))
fig_cost.suptitle("Compute Cost vs Accuracy: Where to Spend Your Budget", fontsize=14)

all_results = []
for name, result in lr_results.items():
    all_results.append(
        {
            "label": f"lr={result['lr']}",
            "time": result["time_sec"],
            "acc": result["accs"][-1],
            "colour": "steelblue",
        }
    )
for name, result in aug_results.items():
    all_results.append(
        {
            "label": name,
            "time": result["time_sec"],
            "acc": result["accs"][-1],
            "colour": "coral",
        }
    )

for r in all_results:
    ax.scatter(r["time"], r["acc"], s=100, c=r["colour"], zorder=5)
    ax.annotate(
        r["label"],
        (r["time"], r["acc"]),
        textcoords="offset points",
        xytext=(10, 5),
        fontsize=8,
    )

ax.set_xlabel("Training Time (seconds)")
ax.set_ylabel("Validation Accuracy")
ax.set_title("Each Point = One Experiment Configuration")
ax.grid(True, alpha=0.3)

sorted_results = sorted(all_results, key=lambda r: r["time"])
pareto_time = [sorted_results[0]["time"]]
pareto_acc = [sorted_results[0]["acc"]]
for r in sorted_results[1:]:
    if r["acc"] > pareto_acc[-1]:
        pareto_time.append(r["time"])
        pareto_acc.append(r["acc"])
if len(pareto_time) > 1:
    ax.plot(pareto_time, pareto_acc, "k--", alpha=0.3, label="Pareto frontier")
    ax.legend()

plt.tight_layout()
plt.savefig("ex_2_04_cost_vs_accuracy.png", dpi=150, bbox_inches="tight")
plt.close(fig_cost)
print("  Saved: ex_2_04_cost_vs_accuracy.png")



### Checkpoint 3: Visualisations generated


In [ ]:
import os

assert os.path.exists(
    "ex_2_04_diminishing_returns.png"
), "Diminishing returns plot missing"
assert os.path.exists("ex_2_04_cost_vs_accuracy.png"), "Cost vs accuracy plot missing"
assert os.path.exists("ex_2_04_lr_sweep_loss.html"), "LR sweep loss plot missing"
print("\n--- Checkpoint 3 passed --- all visualisations saved\n")



## PHASE 5 — APPLY: "How Much Compute Budget Should We Allocate?"


EXPERIMENT RESULTS SUMMARY:
  ═══════════════════════════════════════════════════════════


KEY FINDINGS:
    Best configuration:  {best_overall[0]} (acc={best_overall[1]:.3f})
    Worst configuration: {worst_overall[0]} (acc={worst_overall[1]:.3f})
    Spread:              {best_overall[1] - worst_overall[1]:.3f} accuracy
    Total experiment time: {total_experiment_time:.0f}s ({total_experiment_hours:.2f} hours)
    Total experiment cost:  ${total_experiment_cost:.2f}

  BUDGET RECOMMENDATION FOR THE ENGINEERING MANAGER:
  ═══════════════════════════════════════════════════════════

  MONTHLY COMPUTE BUDGET BREAKDOWN:

  1. Weekly model retraining (production):
     {RETRAIN_EPOCHS} epochs x {RETRAIN_PER_MONTH} retrains/month
     Cost: ${monthly_retrain_cost:,.2f}/month

  2. Monthly hyperparameter search:
     10 configs x {HP_EPOCHS} epochs per search
     Cost: ${hp_search_cost:,.2f}/month

  3. Ad-hoc experiments and debugging:
     Estimated: ${monthly_retrain_cost * 0.5:,.2f}/month

  TOTAL RECOMMENDED BUDGET: ${monthly_retrain_cost + hp_search_cost + monthly_retrain_cost * 0.5:,.2f}/month

  ANSWER TO "Why not $20,000/month?":
    The accuracy spread between our best and worst configs is only
    {best_overall[1] - worst_overall[1]:.3f}. Spending 4x more on training will not
    improve accuracy 4x -- diminishing returns means the extra $15,000
    buys approximately {(best_overall[1] - worst_overall[1]) * 0.2:.3f} additional accuracy.

    Instead, that $15,000 would be better spent on:
    - Better training DATA (more product images, especially edge cases)
    - Human review for the low-confidence predictions (see 01_simple_cnn.py)
    - Monitoring and drift detection (kailash-ml DriftMonitor)

  STAKEHOLDER-READY SUMMARY:
    "Based on systematic experiments tracked with ExperimentTracker,
    the optimal training budget is approximately
    ${monthly_retrain_cost + hp_search_cost + monthly_retrain_cost * 0.5:,.0f}/month.
    This covers weekly retraining, monthly hyperparameter tuning,
    and ad-hoc experiments. The data shows diminishing returns
    beyond this -- additional compute buys less than 1% accuracy
    improvement. The remaining budget is better invested in
    data quality and monitoring infrastructure."


In [ ]:
# SCENARIO: The engineering manager at the Singapore e-commerce platform
# (from 01_simple_cnn.py and 03_production_pipeline.py) asks:
#
#   "We have a $5,000/month compute budget for ML training. The data
#    science team says they need $20,000/month for 'proper' training.
#    How do I know what's actually necessary?"
#
# This is the question hyperparameter studies answer with DATA, not
# opinions. The ExperimentTracker records tell the story.

print("=" * 70)
print("  PHASE 5 — APPLY: Compute Budget Allocation")
print("=" * 70)

# TODO: Calculate actual compute costs from your experiments
#   GPU_COST_PER_HOUR = 3.06  (AWS p3.2xlarge in ap-southeast-1)
#   total_experiment_time = sum of all experiment times
#   total_experiment_hours = total_experiment_time / 3600
#   total_experiment_cost = total_experiment_hours * GPU_COST_PER_HOUR
GPU_COST_PER_HOUR = 3.06

total_experiment_time = ____
total_experiment_hours = total_experiment_time / 3600
total_experiment_cost = total_experiment_hours * GPU_COST_PER_HOUR

# Best and worst results
all_accs = [(name, r["accs"][-1]) for name, r in lr_results.items()]
all_accs += [(name, r["accs"][-1]) for name, r in aug_results.items()]
best_overall = max(all_accs, key=lambda x: x[1])
worst_overall = min(all_accs, key=lambda x: x[1])

# TODO: Project production costs
#   RETRAIN_EPOCHS = 20, RETRAIN_PER_MONTH = 4 (weekly)
#   avg_time_per_epoch = total_experiment_time / (total number of epochs across all runs)
#   retrain_time_hours = (RETRAIN_EPOCHS * avg_time_per_epoch) / 3600
#   retrain_cost_per_run = retrain_time_hours * GPU_COST_PER_HOUR
#   monthly_retrain_cost = retrain_cost_per_run * RETRAIN_PER_MONTH
#   hp_search_time_hours = (10 configs * HP_EPOCHS * avg_time_per_epoch) / 3600
#   hp_search_cost = hp_search_time_hours * GPU_COST_PER_HOUR
RETRAIN_EPOCHS = 20
RETRAIN_PER_MONTH = 4
avg_time_per_epoch = total_experiment_time / (
    len(lr_results) * HP_EPOCHS + len(aug_results) * HP_EPOCHS
)
retrain_time_hours = ____
retrain_cost_per_run = ____
monthly_retrain_cost = ____

hp_search_time_hours = ____
hp_search_cost = ____

print(
)

for name, result in lr_results.items():
    print(
        f"    lr={result['lr']}: acc={result['accs'][-1]:.3f} ({result['time_sec']:.0f}s)"
    )

print(
)

for name, result in aug_results.items():
    print(f"    {name}: acc={result['accs'][-1]:.3f} ({result['time_sec']:.0f}s)")

print(
)



### Checkpoint 4: Apply section complete


In [ ]:
assert total_experiment_cost > 0, "Experiment cost should be positive"
assert best_overall[1] > worst_overall[1], "Best should beat worst"
print("--- Checkpoint 4 passed --- budget analysis complete\n")



## Experiment Summary (all tracked runs)


In [ ]:
print("=" * 70)
print("  FULL EXPERIMENT SUMMARY (all ExperimentTracker runs)")
print("=" * 70)
print(f"  Experiment: {exp_name}")
print(f"  Total runs: {len(lr_results) + len(aug_results)}")
print(
    f"  Total compute: {total_experiment_time:.0f}s ({total_experiment_hours:.2f} hours)"
)
print(f"  Estimated cost: ${total_experiment_cost:.2f}")
print(f"\n  All runs are queryable via ExperimentTracker for future comparison.")



## Clean up


In [ ]:
asyncio.run(conn.close())



## REFLECTION


THEORY:
  [x] Learning rate is the most important hyperparameter -- too high
      causes oscillation, too low wastes compute
  [x] Data augmentation teaches invariance (flip, crop) and improves
      generalisation at zero data collection cost
  [x] Diminishing returns: each doubling of compute buys progressively
      less accuracy improvement

  BUILD + TRAIN:
  [x] Learning rate sweep: {len(LR_SWEEP)} values ({', '.join(str(lr) for lr in LR_SWEEP)})
      Best: lr={best_lr} (acc={best_lr_acc:.3f})
  [x] Augmentation comparison: no_aug vs flip+crop
      Improvement: {aug_improvement:+.3f} accuracy from augmentation
  [x] All {len(lr_results) + len(aug_results)} runs tracked in ExperimentTracker

  VISUALISE (the proof):
  [x] Diminishing returns chart: marginal accuracy gain per epoch
  [x] Cost-vs-accuracy scatter: Pareto frontier of configurations
  [x] Training curves for all LR and augmentation experiments

  APPLY:
  [x] Answered "how much compute budget?" with data, not opinions
  [x] Monthly budget: ${monthly_retrain_cost + hp_search_cost + monthly_retrain_cost * 0.5:,.0f} (vs $20,000 requested)
  [x] Where to invest the savings: data quality > more compute
  [x] Stakeholder-ready summary with dollar values

  KEY INSIGHT: Hyperparameter tuning is not about finding the "perfect"
  setting -- it is about finding the KNEE of the diminishing returns
  curve and allocating budget accordingly. ExperimentTracker makes this
  a data-driven decision instead of a debate between data scientists
  who always want more GPUs and managers who always want lower costs.

  EXERCISE 2 COMPLETE: You've built CNNs (01), added residual connections
  and attention (02), deployed to production (03), and justified the
  compute budget with data (04). This is the full lifecycle of a
  production computer vision system.

  Next: Exercise 3 takes you into sequence modelling with RNNs, LSTMs,
  and GRUs on real Singapore stock-market data...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)



Run one LR sweep trial, logged under its own tracker run.


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 3 — TRAIN: Systematic Hyperparameter Experiments")
print("=" * 70)

# Load data
X_train, y_train, X_val, y_val, train_loader, val_loader = load_cifar10()
conn, tracker, exp_name, registry, has_registry = init_engines()

# ──────────────────────────────────────────────────────────────────────
# EXPERIMENT A: Learning Rate Sweep
# ──────────────────────────────────────────────────────────────────────
LR_SWEEP = [5e-4, 1e-3, 3e-3]
HP_EPOCHS = 6  # Fewer epochs -- enough to see the trend
lr_results: dict[str, dict] = {}

print(
    f"\n  EXPERIMENT A: Learning Rate Sweep ({len(LR_SWEEP)} values, {HP_EPOCHS} epochs each)"
)
print("  " + "-" * 60)


async def train_lr_sweep_async(lr: float) -> tuple[list[float], list[float]]:
    hp_model = ResNetSE()
    lit = LitCNN(hp_model, lr=lr)
    trainer = pl.Trainer(
        max_epochs=HP_EPOCHS,
        accelerator=ACCELERATOR,
        precision=PRECISION,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        enable_checkpointing=False,
    )

    async with tracker.run(
        experiment_name=exp_name, run_name=f"hp_sweep_lr_{lr}"
    ) as ctx:
        await ctx.log_params(
            {
                "architecture": "ResNetSE",
                "lr": str(lr),
                "epochs": str(HP_EPOCHS),
                "sweep_type": "learning_rate",
                "augmentation": "none",
            }
        )

        trainer.fit(lit, train_loader, val_loader)

        for epoch_idx, loss in enumerate(lit.train_losses):
            await ctx.log_metric("train_loss", loss, step=epoch_idx + 1)
        for epoch_idx, acc in enumerate(lit.val_accs):
            await ctx.log_metric("val_accuracy", acc, step=epoch_idx + 1)

        await ctx.log_metrics(
            {
                "final_train_loss": lit.train_losses[-1],
                "final_val_accuracy": lit.val_accs[-1],
            }
        )

    return lit.train_losses, lit.val_accs


for lr in LR_SWEEP:
    name = f"resnet_se_lr{lr}"
    print(f"\n  Training ResNetSE with lr={lr}...")
    t0 = time.perf_counter()
    sweep_losses, sweep_accs  = await train_lr_sweep_async(lr)
    elapsed = time.perf_counter() - t0
    lr_results[name] = {
        "lr": lr,
        "losses": sweep_losses,
        "accs": sweep_accs,
        "time_sec": elapsed,
    }
    print(
        f"    lr={lr}: final_loss={sweep_losses[-1]:.4f}, "
        f"val_acc={sweep_accs[-1]:.3f}, time={elapsed:.1f}s"
    )

# Print LR comparison table
print(f"\n  {'Learning Rate':>15} {'Final Loss':>12} {'Val Acc':>10} {'Time (s)':>10}")
print("  " + "-" * 50)
for name, result in lr_results.items():
    print(
        f"  {result['lr']:>15.4f} {result['losses'][-1]:>12.4f} "
        f"{result['accs'][-1]:>9.3f} {result['time_sec']:>9.1f}"
    )

best_lr_config = max(lr_results.items(), key=lambda x: x[1]["accs"][-1])
best_lr = best_lr_config[1]["lr"]
best_lr_acc = best_lr_config[1]["accs"][-1]
print(f"\n  Best LR: {best_lr} (val_acc={best_lr_acc:.3f})")



### Checkpoint 1: LR sweep complete


Apply CIFAR-10 normalisation to a batch.


Wraps a torchvision dataset to add CIFAR-10 normalisation.


In [ ]:
assert len(lr_results) == len(
    LR_SWEEP
), f"Should have results for all {len(LR_SWEEP)} learning rates"
for name, result in lr_results.items():
    assert result["accs"][-1] > 0.2, (
        f"{name} val_acc={result['accs'][-1]:.3f} is too low -- even suboptimal "
        "LR should learn basic features from 50K images"
    )
print("\n--- Checkpoint 1 passed --- learning rate sweep complete\n")


# ──────────────────────────────────────────────────────────────────────
# EXPERIMENT B: Data Augmentation Comparison
# ──────────────────────────────────────────────────────────────────────
# Data augmentation creates synthetic training variety:
#   - Horizontal flip: a cat facing left is still a cat facing right
#   - Random crop: the object can appear at any position
#   - These teach the model to recognise the OBJECT, not the specific
#     photo composition
#
# We compare: (1) no augmentation vs (2) flip + random crop

print("  EXPERIMENT B: Data Augmentation Comparison")
print("  " + "-" * 60)

# Create augmented training data
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Augmented transform: random horizontal flip + random crop with padding
augmented_transform = T.Compose(
    [
        T.RandomHorizontalFlip(p=0.5),
        T.RandomCrop(32, padding=4),
        T.ToTensor(),
    ]
)

no_aug_transform = T.Compose(
    [
        T.ToTensor(),
    ]
)

# Load raw CIFAR-10 with augmentation transforms applied per-batch
train_set_aug = torchvision.datasets.CIFAR10(
    root=str(DATA_DIR),
    train=True,
    download=True,
    transform=augmented_transform,
)
train_set_noaug = torchvision.datasets.CIFAR10(
    root=str(DATA_DIR),
    train=True,
    download=True,
    transform=no_aug_transform,
)

# For augmented data, we use the dataset directly (augmentation is random
# per-access, so DataLoader applies it on-the-fly each epoch)


def normalise_batch(batch_imgs: torch.Tensor) -> torch.Tensor:
    return (batch_imgs - CIFAR_MEAN) / CIFAR_STD


class NormalisingDataset(torch.utils.data.Dataset):

    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        # img is already a tensor from ToTensor(); normalise it
        img = (img - CIFAR_MEAN.squeeze(0)) / CIFAR_STD.squeeze(0)
        return img, label


aug_loader = DataLoader(
    NormalisingDataset(train_set_aug),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
noaug_loader = DataLoader(
    NormalisingDataset(train_set_noaug),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

# Train with best LR from sweep
aug_results: dict[str, dict] = {}

for aug_name, loader in [("no_augmentation", noaug_loader), ("flip_crop", aug_loader)]:
    print(f"\n  Training ResNetSE with {aug_name} (lr={best_lr})...")
    model = ResNetSE()
    t0 = time.perf_counter()
    losses, accs = train_model(
        model,
        f"aug_{aug_name}",
        tracker,
        exp_name,
        loader,
        val_loader,
        lr=best_lr,
        epochs=HP_EPOCHS,
    )
    elapsed = time.perf_counter() - t0
    aug_results[aug_name] = {
        "losses": losses,
        "accs": accs,
        "time_sec": elapsed,
        "model": model,
    }
    print(
        f"    {aug_name}: final_loss={losses[-1]:.4f}, "
        f"val_acc={accs[-1]:.3f}, time={elapsed:.1f}s"
    )
    # Quick diagnostic check per HP configuration — surfaces whether
    # a high-loss config is hurting the network's CLINICAL HEALTH
    # (dead neurons, vanishing gradients) or merely its accuracy.

    print(f"  ── Diagnostic Report ({aug_name}) ──")
    _diag, _findings = diagnose_classifier(
        model,
        val_loader,
        title=f"ResNetSE {aug_name}",
        n_batches=4,  # brief sweep — full report on winning model
        train_losses=losses,
        val_losses=[1.0 - a for a in accs],
        show=False,
    )
    # ══════ EXPECTED OUTPUT (synthesized reference across HP configs) ══════
    # Typical Prescription-Pad patterns observed per augmentation config:
    # ┌──────────────────────────┬──────────────────────────────────────────┐
    # │ Config                   │ Typical Prescription Pad finding         │
    # ├──────────────────────────┼──────────────────────────────────────────┤
    # │ no_augmentation          │ [!] Overfitting: train-val gap ~18%.     │
    # │                          │     Val loss starts rising after ep 5.   │
    # │                          │     Dead neurons: 4% (healthy).          │
    # │                          │     Val acc: ~0.55                        │
    # ├──────────────────────────┼──────────────────────────────────────────┤
    # │ flip_only                │ [✓] Mildly overfitting: gap ~12%.        │
    # │                          │     Val acc: ~0.58                        │
    # ├──────────────────────────┼──────────────────────────────────────────┤
    # │ flip_crop (winner)       │ [✓] All HEALTHY. Gap ~6%. Val acc: ~0.62 │
    # ├──────────────────────────┼──────────────────────────────────────────┤
    # │ strong_augment           │ [!] Underfitting: train loss plateau.    │
    # │                          │     Val acc: ~0.54 (below flip_crop)     │
    # └──────────────────────────┴──────────────────────────────────────────┘
    #
    # STUDENT INTERPRETATION GUIDE — reading HP diagnostics:
    #
    #  [STETHOSCOPE — HP SIGNATURE READING] Each HP config
    #     produces a DIFFERENT pathology pattern. no_augmentation
    #     shows the classic overfitting U-curve in val loss (val
    #     rising while train falls). strong_augment shows the
    #     opposite: underfitting (train plateau because the data
    #     looks different every batch and the model cannot
    #     converge on a single distribution). flip_crop hits the
    #     sweet spot — regularisation without underfitting.
    #     Slide 5R covers the "augmentation strength dial": from
    #     zero (overfit) → flip_crop (optimal) → strong (underfit).
    #     >> Prescription: If every HP config is overfitting,
    #        increase augmentation. If every config is
    #        underfitting, decrease augmentation. Look for the
    #        pattern ACROSS configs, not within one config.
    #
    #  [X-RAY — HP-INDUCED DEAD NEURONS] A too-aggressive LR or
    #     too-strong augmentation can spike dead-neuron fractions
    #     15-30% above baseline. If you see WARNING findings
    #     only in the HIGH-LR configs, the gradient updates are
    #     saturating ReLUs into permanent death. This is a
    #     SEPARATE diagnostic from accuracy — a config can lose
    #     accuracy for many reasons, but high dead% narrows the
    #     cause to optimisation dynamics.
    #     >> Prescription: Reduce LR to 3e-4, add warmup schedule
    #        (0 → LR over first 500 steps) to avoid the early-
    #        training dead-ReLU spike.
    #
    #  [BLOOD TEST — CROSS-CONFIG COMPARISON] Gradient RMS across
    #     HP configs should stay within 10x of each other. If
    #     one config has RMS <1e-5 while others are ~1e-3, that
    #     config has broken optimisation (usually LR far too
    #     small or gradient clipping too tight).
    #     >> Prescription: Plot min RMS per config as a bar
    #        chart. Outliers indicate hyperparameters breaking
    #        the backward pass.
    #
    #  FIVE-INSTRUMENT TAKEAWAY: HP sweeps are FIVE parallel
    #  diagnostic experiments. The Prescription Pad tells you
    #  WHY a config failed — accuracy numbers alone tell you
    #  only THAT it failed. This lifts HP tuning from pure trial-
    #  and-error to clinical reasoning. You'll apply the same
    #  discipline to LR schedule sweeps in ex_4 transformers and
    #  to entropy-coefficient sweeps in ex_8 RL.
    # ═════════════════════════════════════════════════════════════════════

# Print augmentation comparison
print(f"\n  {'Augmentation':>20} {'Final Loss':>12} {'Val Acc':>10} {'Time (s)':>10}")
print("  " + "-" * 55)
for name, result in aug_results.items():
    print(
        f"  {name:>20} {result['losses'][-1]:>12.4f} "
        f"{result['accs'][-1]:>9.3f} {result['time_sec']:>9.1f}"
    )

aug_improvement = (
    aug_results["flip_crop"]["accs"][-1] - aug_results["no_augmentation"]["accs"][-1]
)
print(f"\n  Augmentation improvement: {aug_improvement:+.3f} accuracy")



### Checkpoint 2: Augmentation comparison complete


In [ ]:
assert len(aug_results) == 2, "Should have results for both augmentation settings"
print("\n--- Checkpoint 2 passed --- augmentation comparison complete\n")



## PHASE 4 — VISUALISE: Accuracy-vs-Cost Tradeoff Curves


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: The Accuracy-vs-Cost Tradeoff")
print("=" * 70)

viz = create_visualizer()

# Plot 1: Learning rate sweep — training losses
save_training_plots(
    viz,
    {f"lr={r['lr']} loss": r["losses"] for r in lr_results.values()},
    "ex_2_04_lr_sweep_loss.html",
    y_label="Training Loss",
)

# Plot 2: Learning rate sweep — validation accuracies
save_training_plots(
    viz,
    {f"lr={r['lr']} acc": r["accs"] for r in lr_results.values()},
    "ex_2_04_lr_sweep_acc.html",
    y_label="Validation Accuracy",
)

# Plot 3: Augmentation comparison
save_training_plots(
    viz,
    {f"{name} loss": r["losses"] for name, r in aug_results.items()},
    "ex_2_04_augmentation_loss.html",
    y_label="Training Loss",
)
save_training_plots(
    viz,
    {f"{name} acc": r["accs"] for name, r in aug_results.items()},
    "ex_2_04_augmentation_acc.html",
    y_label="Validation Accuracy",
)

# Plot 4: The "diminishing returns" chart — accuracy at each epoch
# This is the KEY visualisation for the engineering manager
fig_roi, axes = plt.subplots(1, 2, figsize=(16, 6))
fig_roi.suptitle(
    "The Diminishing Returns of Training: When to Stop Spending",
    fontsize=14,
)

# Left: Accuracy vs epoch for all LR configs
for name, result in lr_results.items():
    epochs_x = list(range(1, len(result["accs"]) + 1))
    axes[0].plot(epochs_x, result["accs"], "o-", label=f"lr={result['lr']}")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Validation Accuracy")
axes[0].set_title("Accuracy vs Training Duration")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Marginal improvement per epoch (the "ROI curve")
# Shows how much each additional epoch buys
for name, result in lr_results.items():
    accs = result["accs"]
    marginal = [accs[0]] + [accs[i] - accs[i - 1] for i in range(1, len(accs))]
    epochs_x = list(range(1, len(marginal) + 1))
    axes[1].bar(
        [x + 0.2 * (list(lr_results.keys()).index(name) - 1) for x in epochs_x],
        marginal,
        width=0.2,
        alpha=0.7,
        label=f"lr={result['lr']}",
    )
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Marginal Accuracy Gain")
axes[1].set_title("Diminishing Returns: Accuracy Gained Per Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color="black", linewidth=0.5)

plt.tight_layout()
plt.savefig("ex_2_04_diminishing_returns.png", dpi=150, bbox_inches="tight")
plt.close(fig_roi)
print("  Saved: ex_2_04_diminishing_returns.png")
print("  Left: accuracy climbs steeply then flattens (the 'knee')")
print("  Right: marginal gains shrink each epoch (the ROI curve)")

# Plot 5: Cost-accuracy tradeoff (compute time vs accuracy)
fig_cost, ax = plt.subplots(1, 1, figsize=(10, 6))
fig_cost.suptitle("Compute Cost vs Accuracy: Where to Spend Your Budget", fontsize=14)

# Each point = one experiment configuration
all_results = []
for name, result in lr_results.items():
    all_results.append(
        {
            "label": f"lr={result['lr']}",
            "time": result["time_sec"],
            "acc": result["accs"][-1],
            "colour": "steelblue",
        }
    )
for name, result in aug_results.items():
    all_results.append(
        {
            "label": name,
            "time": result["time_sec"],
            "acc": result["accs"][-1],
            "colour": "coral",
        }
    )

for r in all_results:
    ax.scatter(r["time"], r["acc"], s=100, c=r["colour"], zorder=5)
    ax.annotate(
        r["label"],
        (r["time"], r["acc"]),
        textcoords="offset points",
        xytext=(10, 5),
        fontsize=8,
    )

ax.set_xlabel("Training Time (seconds)")
ax.set_ylabel("Validation Accuracy")
ax.set_title("Each Point = One Experiment Configuration")
ax.grid(True, alpha=0.3)

# Add a "Pareto frontier" line connecting non-dominated points
sorted_results = sorted(all_results, key=lambda r: r["time"])
pareto_time = [sorted_results[0]["time"]]
pareto_acc = [sorted_results[0]["acc"]]
for r in sorted_results[1:]:
    if r["acc"] > pareto_acc[-1]:
        pareto_time.append(r["time"])
        pareto_acc.append(r["acc"])
if len(pareto_time) > 1:
    ax.plot(pareto_time, pareto_acc, "k--", alpha=0.3, label="Pareto frontier")
    ax.legend()

plt.tight_layout()
plt.savefig("ex_2_04_cost_vs_accuracy.png", dpi=150, bbox_inches="tight")
plt.close(fig_cost)
print("  Saved: ex_2_04_cost_vs_accuracy.png")



### Checkpoint 3: Visualisations generated


In [ ]:
import os

assert os.path.exists(
    "ex_2_04_diminishing_returns.png"
), "Diminishing returns plot missing"
assert os.path.exists("ex_2_04_cost_vs_accuracy.png"), "Cost vs accuracy plot missing"
assert os.path.exists("ex_2_04_lr_sweep_loss.html"), "LR sweep loss plot missing"
print("\n--- Checkpoint 3 passed --- all visualisations saved\n")



## PHASE 5 — APPLY: "How Much Compute Budget Should We Allocate?"


EXPERIMENT RESULTS SUMMARY:
  ═══════════════════════════════════════════════════════════


KEY FINDINGS:
    Best configuration:  {best_overall[0]} (acc={best_overall[1]:.3f})
    Worst configuration: {worst_overall[0]} (acc={worst_overall[1]:.3f})
    Spread:              {best_overall[1] - worst_overall[1]:.3f} accuracy
    Total experiment time: {total_experiment_time:.0f}s ({total_experiment_hours:.2f} hours)
    Total experiment cost:  ${total_experiment_cost:.2f}

  BUDGET RECOMMENDATION FOR THE ENGINEERING MANAGER:
  ═══════════════════════════════════════════════════════════

  MONTHLY COMPUTE BUDGET BREAKDOWN:

  1. Weekly model retraining (production):
     {RETRAIN_EPOCHS} epochs x {RETRAIN_PER_MONTH} retrains/month
     Cost: ${monthly_retrain_cost:,.2f}/month

  2. Monthly hyperparameter search:
     10 configs x {HP_EPOCHS} epochs per search
     Cost: ${hp_search_cost:,.2f}/month

  3. Ad-hoc experiments and debugging:
     Estimated: ${monthly_retrain_cost * 0.5:,.2f}/month

  TOTAL RECOMMENDED BUDGET: ${monthly_retrain_cost + hp_search_cost + monthly_retrain_cost * 0.5:,.2f}/month

  ANSWER TO "Why not $20,000/month?":
    The accuracy spread between our best and worst configs is only
    {best_overall[1] - worst_overall[1]:.3f}. Spending 4x more on training will not
    improve accuracy 4x -- diminishing returns means the extra $15,000
    buys approximately {(best_overall[1] - worst_overall[1]) * 0.2:.3f} additional accuracy.

    Instead, that $15,000 would be better spent on:
    - Better training DATA (more product images, especially edge cases)
    - Human review for the low-confidence predictions (see 01_simple_cnn.py)
    - Monitoring and drift detection (kailash-ml DriftMonitor)

  STAKEHOLDER-READY SUMMARY:
    "Based on systematic experiments tracked with ExperimentTracker,
    the optimal training budget is approximately
    ${monthly_retrain_cost + hp_search_cost + monthly_retrain_cost * 0.5:,.0f}/month.
    This covers weekly retraining, monthly hyperparameter tuning,
    and ad-hoc experiments. The data shows diminishing returns
    beyond this -- additional compute buys less than 1% accuracy
    improvement. The remaining budget is better invested in
    data quality and monitoring infrastructure."


In [ ]:
# SCENARIO: The engineering manager at the Singapore e-commerce platform
# (from 01_simple_cnn.py and 03_production_pipeline.py) asks:
#
#   "We have a $5,000/month compute budget for ML training. The data
#    science team says they need $20,000/month for 'proper' training.
#    How do I know what's actually necessary?"
#
# This is the question hyperparameter studies answer with DATA, not
# opinions. The ExperimentTracker records tell the story.

print("=" * 70)
print("  PHASE 5 — APPLY: Compute Budget Allocation")
print("=" * 70)

# Calculate actual compute costs from our experiments
# AWS p3.2xlarge (V100 GPU): $3.06/hr in ap-southeast-1 (Singapore)
GPU_COST_PER_HOUR = 3.06

total_experiment_time = sum(r["time_sec"] for r in lr_results.values())
total_experiment_time += sum(r["time_sec"] for r in aug_results.values())
total_experiment_hours = total_experiment_time / 3600
total_experiment_cost = total_experiment_hours * GPU_COST_PER_HOUR

# Best result from all experiments
all_accs = [(name, r["accs"][-1]) for name, r in lr_results.items()]
all_accs += [(name, r["accs"][-1]) for name, r in aug_results.items()]
best_overall = max(all_accs, key=lambda x: x[1])
worst_overall = min(all_accs, key=lambda x: x[1])

# Projected costs at production scale
# Assume: retrain weekly, full 50K dataset, 20 epochs per retrain
RETRAIN_EPOCHS = 20
RETRAIN_PER_MONTH = 4  # weekly
avg_time_per_epoch = total_experiment_time / (
    len(lr_results) * HP_EPOCHS + len(aug_results) * HP_EPOCHS
)
retrain_time_hours = (RETRAIN_EPOCHS * avg_time_per_epoch) / 3600
retrain_cost_per_run = retrain_time_hours * GPU_COST_PER_HOUR
monthly_retrain_cost = retrain_cost_per_run * RETRAIN_PER_MONTH

# Hyperparameter search cost (monthly)
# 10 configurations, 6 epochs each
hp_search_time_hours = (10 * HP_EPOCHS * avg_time_per_epoch) / 3600
hp_search_cost = hp_search_time_hours * GPU_COST_PER_HOUR

print(
)

for name, result in lr_results.items():
    print(
        f"    lr={result['lr']}: acc={result['accs'][-1]:.3f} ({result['time_sec']:.0f}s)"
    )

print(
)

for name, result in aug_results.items():
    print(f"    {name}: acc={result['accs'][-1]:.3f} ({result['time_sec']:.0f}s)")

print(
)



### Checkpoint 4: Apply section complete


In [ ]:
assert total_experiment_cost > 0, "Experiment cost should be positive"
assert best_overall[1] > worst_overall[1], "Best should beat worst"
print("--- Checkpoint 4 passed --- budget analysis complete\n")



## Experiment Summary (all tracked runs)


In [ ]:
print("=" * 70)
print("  FULL EXPERIMENT SUMMARY (all ExperimentTracker runs)")
print("=" * 70)
print(f"  Experiment: {exp_name}")
print(f"  Total runs: {len(lr_results) + len(aug_results)}")
print(
    f"  Total compute: {total_experiment_time:.0f}s ({total_experiment_hours:.2f} hours)"
)
print(f"  Estimated cost: ${total_experiment_cost:.2f}")
print(f"\n  All runs are queryable via ExperimentTracker for future comparison.")



## Clean up


In [ ]:
asyncio.run(conn.close())



## REFLECTION


THEORY:
  [x] Learning rate is the most important hyperparameter -- too high
      causes oscillation, too low wastes compute
  [x] Data augmentation teaches invariance (flip, crop) and improves
      generalisation at zero data collection cost
  [x] Diminishing returns: each doubling of compute buys progressively
      less accuracy improvement

  BUILD + TRAIN:
  [x] Learning rate sweep: {len(LR_SWEEP)} values ({', '.join(str(lr) for lr in LR_SWEEP)})
      Best: lr={best_lr} (acc={best_lr_acc:.3f})
  [x] Augmentation comparison: no_aug vs flip+crop
      Improvement: {aug_improvement:+.3f} accuracy from augmentation
  [x] All {len(lr_results) + len(aug_results)} runs tracked in ExperimentTracker

  VISUALISE (the proof):
  [x] Diminishing returns chart: marginal accuracy gain per epoch
  [x] Cost-vs-accuracy scatter: Pareto frontier of configurations
  [x] Training curves for all LR and augmentation experiments

  APPLY:
  [x] Answered "how much compute budget?" with data, not opinions
  [x] Monthly budget: ${monthly_retrain_cost + hp_search_cost + monthly_retrain_cost * 0.5:,.0f} (vs $20,000 requested)
  [x] Where to invest the savings: data quality > more compute
  [x] Stakeholder-ready summary with dollar values

  KEY INSIGHT: Hyperparameter tuning is not about finding the "perfect"
  setting -- it is about finding the KNEE of the diminishing returns
  curve and allocating budget accordingly. ExperimentTracker makes this
  a data-driven decision instead of a debate between data scientists
  who always want more GPUs and managers who always want lower costs.

  EXERCISE 2 COMPLETE: You've built CNNs (01), added residual connections
  and attention (02), deployed to production (03), and justified the
  compute budget with data (04). This is the full lifecycle of a
  production computer vision system.

  Next: Exercise 3 takes you into sequence modelling with RNNs, LSTMs,
  and GRUs on real Singapore stock-market data...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

